In [1]:
# --- CELL 0: Mount & install deps ---
from google.colab import drive
drive.mount('/content/drive')

!pip -q install lightgbm==4.5.0 pandas numpy matplotlib joblib


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 21.7 MB/s eta 0:00:00


In [2]:
# --- CELL 1: CONFIG ---
import os, glob, json, joblib, pandas as pd, numpy as np

# 👇 EDIT THIS PATH ONLY if your folder differs
BASE = "/content/drive/MyDrive/JEFF UNI/week 6/Collab"

# Inputs
FEATURES_CSV = f"{BASE}/nsw_demand_features.csv"
COHORT = {
    "overall_all": f"{BASE}/3cohort_overall_all.csv",
    "hotday_all":  f"{BASE}/3cohort_hotday_all.csv",
}

# Models
MODELS_LGBM_DIR     = f"{BASE}/models_lgbm_48to48"
MODELS_LGBMALT_DIR  = f"{BASE}/models_lgbmALT_48to48"  # optional

# Outputs
PRED_DIR   = f"{BASE}/Cohort3-Predictions"
METRICS_DIR= f"{BASE}/Cohort3-Metrics"
FIG_DIR    = f"{BASE}/Cohort3-Figures"
for d in [PRED_DIR, METRICS_DIR, FIG_DIR]:
    os.makedirs(d, exist_ok=True)

# Controls
HORIZON = 48
TRAIN_MISSING_BLOCKS = False   # <- keep False unless you MUST retrain missing blocks

print("BASE:", BASE)
print("FEATURES_CSV:", FEATURES_CSV)
print("COHORT files:", COHORT)
print("LGBM models dir:", MODELS_LGBM_DIR)
print("ALT  models dir:", MODELS_LGBMALT_DIR)
print("Outputs:", {"pred": PRED_DIR, "metrics": METRICS_DIR, "figs": FIG_DIR})


BASE: /content/drive/MyDrive/JEFF UNI/week 6/Collab
FEATURES_CSV: /content/drive/MyDrive/JEFF UNI/week 6/Collab/nsw_demand_features.csv
COHORT files: {'overall_all': '/content/drive/MyDrive/JEFF UNI/week 6/Collab/3cohort_overall_all.csv', 'hotday_all': '/content/drive/MyDrive/JEFF UNI/week 6/Collab/3cohort_hotday_all.csv'}
LGBM models dir: /content/drive/MyDrive/JEFF UNI/week 6/Collab/models_lgbm_48to48
ALT  models dir: /content/drive/MyDrive/JEFF UNI/week 6/Collab/models_lgbmALT_48to48
Outputs: {'pred': '/content/drive/MyDrive/JEFF UNI/week 6/Collab/Cohort3-Predictions', 'metrics': '/content/drive/MyDrive/JEFF UNI/week 6/Collab/Cohort3-Metrics', 'figs': '/content/drive/MyDrive/JEFF UNI/week 6/Collab/Cohort3-Figures'}


In [3]:
# --- CELL 2: HELPERS (robust detection) ---
import warnings
warnings.filterwarnings("ignore")

def first_present(cols, options):
    for c in options:
        if c in cols: return c
    return None

def detect_key_column(df_a, df_b):
    candidates = ["trading_interval","ts","SETTLEMENTDATE","datetime","timestamp","time","TradingInterval"]
    a = set(df_a.columns); b = set(df_b.columns)
    both = [c for c in candidates if c in a and c in b]
    if both:
        return both[0]
    # fallback: intersection of object/datetime columns
    common = list(a & b)
    for c in common:
        if np.issubdtype(df_a[c].dtype, np.datetime64) or df_a[c].dtype==object:
            return c
    raise ValueError("Could not detect a common key column. Please align cohort & features time column names.")

def coerce_datetime(series):
    try:
        return pd.to_datetime(series, errors="coerce", utc=True)
    except Exception:
        return pd.to_datetime(series, errors="coerce")

def pick_target_col(features_df):
    choices = ["y","target","demand","TOTALDEMAND","operational_demand","OPERATIONAL_DEMAND"]
    c = first_present(features_df.columns, choices)
    if not c:
        # last numeric column heuristic
        num = features_df.select_dtypes(include=[np.number]).columns.tolist()
        if not num: raise ValueError("No numeric target column found in features.")
        c = num[-1]
    return c

def pick_operator_col(cohort_df):
    return first_present(cohort_df.columns, ["op_24h_latest","op_latest","operator","Operator","OPERATOR"])

def load_meta(models_dir):
    meta_path = os.path.join(models_dir, "meta.json")
    if os.path.exists(meta_path):
        with open(meta_path,"r") as f:
            meta = json.load(f)
        feat_cols = meta.get("feat_cols")
        blocks = meta.get("blocks", [[1,12],[13,24],[25,36],[37,48]])
        return feat_cols, blocks
    return None, [[1,12],[13,24],[25,36],[37,48]]

def load_block_models(models_dir):
    if not os.path.isdir(models_dir):
        return []
    jobs = sorted(glob.glob(os.path.join(models_dir, "*.joblib")))
    models = []
    for j in jobs:
        try:
            models.append(joblib.load(j))
        except Exception as e:
            print("Skipping unreadable model:", j, str(e))
    return models

def ensure_feat_cols(features_df, key_col, target_col, feat_cols_from_meta):
    if feat_cols_from_meta:
        missing = [c for c in feat_cols_from_meta if c not in features_df.columns]
        if missing:
            raise ValueError(f"Features missing columns from meta: {missing}")
        return feat_cols_from_meta
    # fallback: all numeric except the target
    num = features_df.select_dtypes(include=[np.number]).columns.tolist()
    feat_cols = [c for c in num if c != target_col]
    return feat_cols

def build_design_for_cohort(cohort_csv, features_df, feat_cols):
    idx = pd.read_csv(cohort_csv)
    key = detect_key_column(idx, features_df)
    idx[key] = coerce_datetime(idx[key])
    features_df = features_df.copy()
    features_df[key] = coerce_datetime(features_df[key])
    m = pd.merge(idx[[key]], features_df[[key]+feat_cols], on=key, how="left").sort_values(key)
    # Safety: drop duplicates on key to avoid reindex error
    m = m.drop_duplicates(subset=[key])
    X = m[feat_cols].astype(float).values
    return X, idx, key

def truths_from_features_for_cohort(cohort_csv, features_df, target_col):
    idx = pd.read_csv(cohort_csv)
    key = detect_key_column(idx, features_df)
    idx[key] = coerce_datetime(idx[key])
    features_df = features_df.copy()
    features_df[key] = coerce_datetime(features_df[key])

    # Outer merge to keep order by cohort index times; then for each start time take the next 48 rows in features timeline
    f_sorted = features_df.sort_values(key).reset_index(drop=True)
    times = idx[key].drop_duplicates().sort_values().values

    # map time -> row index in features (nearest exact match)
    f_map = pd.Series(range(len(f_sorted)), index=f_sorted[key].values)
    rows = []
    for t in times:
        if pd.isna(t) or t not in f_map.index:
            continue
        start = int(f_map.loc[t])
        y_window = f_sorted[target_col].iloc[start:start+HORIZON].values
        if len(y_window)==HORIZON:
            rows.append(y_window)
    if not rows:
        raise ValueError("Could not align truths; check timestamp alignment between cohort and features.")
    Y = np.vstack(rows)  # (N,48)
    return Y

def compute_metrics(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    err = y_pred - y_true
    mae = np.mean(np.abs(err))
    rmse = np.sqrt(np.mean(err**2))
    with np.errstate(divide='ignore', invalid='ignore'):
        mape = np.mean(np.abs(err) / np.maximum(1e-8, np.abs(y_true))) * 100.0
    bias = np.mean(err)
    return {"MAE": float(mae), "RMSE": float(rmse), "MAPE": float(mape), "Bias": float(bias)}


In [5]:
# --- CELL 3 (FIXED): LGBM MODEL LOAD / (optional) TRAIN MISSING BLOCKS ---
from lightgbm import LGBMRegressor

feat_cols_meta, blocks = load_meta(MODELS_LGBM_DIR)

# Load features once
features_df = pd.read_csv(FEATURES_CSV)

# Pick target, then derive feature columns (or use meta)
target_col = pick_target_col(features_df)
feat_cols = ensure_feat_cols(features_df, None, target_col, feat_cols_meta)

print(f"Detected target: {target_col}")
print(f"#feat_cols: {len(feat_cols)}")
print("Blocks:", blocks)

# (Optional) show candidate time columns present in FEATURES for your info
_candidates = ["trading_interval","ts","SETTLEMENTDATE","datetime","timestamp","time","TradingInterval"]
print("Possible time columns in FEATURES:", [c for c in _candidates if c in features_df.columns])

# Load saved block models (no retrain unless you enable it)
lgbm_models = load_block_models(MODELS_LGBM_DIR)
print(f"Loaded LGBM block models: {len(lgbm_models)}")

if TRAIN_MISSING_BLOCKS and len(lgbm_models) < 4:
    print("TRAIN_MISSING_BLOCKS=True and some blocks missing → training those blocks now.")
    os.makedirs(MODELS_LGBM_DIR, exist_ok=True)
    # Minimal example training per horizon (only for missing blocks)
    keycol = first_present(features_df.columns, _candidates)
    features_df = features_df.sort_values(keycol if keycol else features_df.columns[0])
    X_full = features_df[feat_cols].astype(float).values
    have = {os.path.basename(p) for p in glob.glob(os.path.join(MODELS_LGBM_DIR,"*.joblib"))}
    for (a,b) in blocks:
        fname = f"lgbm_block_{a}_{b}.joblib"
        if fname in have:
            continue
        sub_models = []
        for h in range(a, b+1):
            y = features_df[target_col].shift(-h).values
            mask = ~np.isnan(y)
            m = LGBMRegressor(n_estimators=400, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8, random_state=42+h)
            m.fit(X_full[mask], y[mask])
            sub_models.append(m)
        joblib.dump(sub_models, os.path.join(MODELS_LGBM_DIR, fname))
    # write meta if missing
    meta_path = os.path.join(MODELS_LGBM_DIR,"meta.json")
    if not os.path.exists(meta_path):
        with open(meta_path,"w") as f:
            json.dump({"feat_cols": feat_cols, "blocks": blocks}, f)
    lgbm_models = load_block_models(MODELS_LGBM_DIR)
    print(f"After training, loaded blocks: {len(lgbm_models)}")


Detected target: y
#feat_cols: 28
Blocks: [[1, 12], [13, 24], [25, 36], [37, 48]]
Possible time columns in FEATURES: []
Loaded LGBM block models: 4


In [6]:
# --- CELL 4: ALT MODELS (optional) + predict function ---
lgbmALT_models = load_block_models(MODELS_LGBMALT_DIR)
print(f"Loaded ALT block models: {len(lgbmALT_models)}")

def predict_48_from_blocks(block_models_list, X):
    """
    block_models_list: list of 4 items, each is either a single model with .predict on 2D,
                       or a list of per-horizon models.
    Returns (N, 48)
    """
    if not block_models_list:
        raise ValueError("No models provided.")
    preds = []
    for bm, (a,b) in zip(block_models_list, blocks):
        # if bm is a list of per-horizon models
        if isinstance(bm, (list, tuple)):
            P = []
            for h_idx, m in enumerate(bm):
                P.append(m.predict(X))
            P = np.column_stack(P)  # (N, block_width)
        else:
            # assume model predicts block width columns
            P = bm.predict(X)
            if P.ndim==1:
                P = P.reshape(-1, b-a+1)
        preds.append(P)
    return np.column_stack(preds)  # (N,48)


Loaded ALT block models: 0


In [9]:
# --- CELL 5A: Robust time alignment (handles different column names) ---
import pandas as pd
import numpy as np

# override: detect the most likely time col in a single df
def _find_time_col(df: pd.DataFrame):
    candidates = ["trading_interval","TradingInterval","SETTLEMENTDATE","ts","datetime","timestamp","time","DateTime"]
    for c in candidates:
        if c in df.columns:
            return c
    # fallback: first datetime-like column by inference
    for c in df.columns:
        s = pd.to_datetime(df[c], errors="coerce", utc=True)
        # if at least half parse, accept
        if s.notna().mean() > 0.5:
            return c
    return None

# NEW: build_design_for_cohort that aligns on time even if names differ
def build_design_for_cohort(cohort_csv, features_df, feat_cols):
    idx = pd.read_csv(cohort_csv)
    feats = features_df.copy()

    key_a = _find_time_col(idx)
    key_b = _find_time_col(feats)
    if key_a is None or key_b is None:
        raise ValueError(f"Could not find a time column. Cohort cols: {list(idx.columns)[:10]} | Features cols: {list(feats.columns)[:10]}")

    # parse to datetime (UTC); keep originals for debugging
    idx[key_a]  = pd.to_datetime(idx[key_a], errors="coerce", utc=True)
    feats[key_b]= pd.to_datetime(feats[key_b], errors="coerce", utc=True)

    # sort & dedup by time
    idx = idx.dropna(subset=[key_a]).drop_duplicates(subset=[key_a]).sort_values(key_a).reset_index(drop=True)
    feats = feats.dropna(subset=[key_b]).drop_duplicates(subset=[key_b]).sort_values(key_b).reset_index(drop=True)

    # Try exact merge first
    exact = pd.merge(idx[[key_a]], feats[[key_b]+feat_cols], left_on=key_a, right_on=key_b, how="left")

    if exact[feat_cols].isna().all(axis=None):
        # No exact matches → use nearest-time match with small tolerance
        # (5 minutes is generous for second/zone shifts but tight enough to avoid wrong days)
        exact = pd.merge_asof(
            idx[[key_a]].sort_values(key_a),
            feats[[key_b]+feat_cols].sort_values(key_b),
            left_on=key_a, right_on=key_b,
            direction="nearest", tolerance=pd.Timedelta("5min")
        )

    # Final safety: drop any remaining rows that failed to align
    aligned = exact.dropna(subset=feat_cols)
    if aligned.empty:
        raise ValueError(
            "Could not align cohort times to features (even with nearest match). "
            f"Example cohort head:\n{idx[[key_a]].head()}\n\n"
            f"Example features head:\n{feats[[key_b]].head()}"
        )

    X = aligned[feat_cols].astype(float).values
    return X, idx, key_a  # we return the cohort time key name for info


In [10]:
# --- CELL 5: PREDICTIONS (LGBM, ALT if present, EVL) ---
import pandas as pd, numpy as np, os

assert len(lgbm_models) > 0, "No LGBM block models loaded."

def predict_and_write(name, cohort_path, model_tag, model_list):
    Xc, idx_df, key_used = build_design_for_cohort(cohort_path, features_df, feat_cols)
    P  = predict_48_from_blocks(model_list, Xc)  # (N,48)
    assert P.shape[1] == HORIZON
    out_csv = f"{PRED_DIR}/predictions__{model_tag}__{name}.csv"
    pd.DataFrame(P).to_csv(out_csv, index=False)
    print(f"[{model_tag}] wrote:", out_csv, P.shape)
    return out_csv

written = {"LGBM":{}, "LGBMALT":{}, "EVL":{}}

for name, path in COHORT.items():
    # LGBM
    lgbm_csv = predict_and_write(name, path, "JEFF__LGBM", lgbm_models)
    written["LGBM"][name] = lgbm_csv

    # ALT (if present)
    have_alt = len(lgbmALT_models) > 0
    if have_alt:
        alt_csv = predict_and_write(name, path, "JEFF__LGBMALT", lgbmALT_models)
        written["LGBMALT"][name] = alt_csv
        # EVL = average
        P_lgb = pd.read_csv(lgbm_csv).values
        P_alt = pd.read_csv(alt_csv).values
        assert P_lgb.shape == P_alt.shape
        P_evl = 0.5 * (P_lgb + P_alt)
        evl_csv = f"{PRED_DIR}/predictions__JEFF__EVL__{name}.csv"
        pd.DataFrame(P_evl).to_csv(evl_csv, index=False)
        print(f"[EVL] wrote:", evl_csv, P_evl.shape)
        written["EVL"][name] = evl_csv
    else:
        print("ALT models not found → skipping ALT & EVL for", name)

print("Prediction files:", written)


[JEFF__LGBM] wrote: /content/drive/MyDrive/JEFF UNI/week 6/Collab/Cohort3-Predictions/predictions__JEFF__LGBM__overall_all.csv (38736, 48)
ALT models not found → skipping ALT & EVL for overall_all
[JEFF__LGBM] wrote: /content/drive/MyDrive/JEFF UNI/week 6/Collab/Cohort3-Predictions/predictions__JEFF__LGBM__hotday_all.csv (3888, 48)
ALT models not found → skipping ALT & EVL for hotday_all
Prediction files: {'LGBM': {'overall_all': '/content/drive/MyDrive/JEFF UNI/week 6/Collab/Cohort3-Predictions/predictions__JEFF__LGBM__overall_all.csv', 'hotday_all': '/content/drive/MyDrive/JEFF UNI/week 6/Collab/Cohort3-Predictions/predictions__JEFF__LGBM__hotday_all.csv'}, 'LGBMALT': {}, 'EVL': {}}


In [12]:
# --- PATCH: robust truths_from_features_for_cohort (name-agnostic + asof align) ---
import pandas as pd
import numpy as np

def _find_time_col(df: pd.DataFrame):
    candidates = ["trading_interval","TradingInterval","SETTLEMENTDATE","ts",
                  "datetime","timestamp","time","DateTime"]
    for c in candidates:
        if c in df.columns: return c
    # fallback: first column that parses to datetime for >50% rows
    for c in df.columns:
        s = pd.to_datetime(df[c], errors="coerce", utc=True)
        if s.notna().mean() > 0.5:
            return c
    return None

def truths_from_features_for_cohort(cohort_csv: str, features_df: pd.DataFrame, target_col: str, horizon:int=48):
    """
    Returns Y_true with shape (N, horizon) for each cohort start time.
    Aligns cohort times to features times by exact match first, then nearest (±5min).
    Drops any rows that fail alignment or don't have a full horizon window.
    """
    idx = pd.read_csv(cohort_csv).copy()
    feats = features_df.copy()

    key_a = _find_time_col(idx)
    key_b = _find_time_col(feats)
    if key_a is None or key_b is None:
        raise ValueError(f"Could not find a time column in cohort({list(idx.columns)[:10]}) "
                         f"or features({list(feats.columns)[:10]}).")

    # normalize times
    idx[key_a]   = pd.to_datetime(idx[key_a], errors="coerce", utc=True)
    feats[key_b] = pd.to_datetime(feats[key_b], errors="coerce", utc=True)

    # clean, sort, dedup
    idx   = idx.dropna(subset=[key_a]).drop_duplicates(subset=[key_a]).sort_values(key_a).reset_index(drop=True)
    feats = feats.dropna(subset=[key_b]).drop_duplicates(subset=[key_b]).sort_values(key_b).reset_index(drop=True)

    # build a position index for features timeline
    feats = feats.sort_values(key_b).reset_index(drop=True)
    feats["__pos__"] = np.arange(len(feats))

    # try exact match first
    exact = pd.merge(idx[[key_a]], feats[[key_b,"__pos__"]], left_on=key_a, right_on=key_b, how="left")

    if exact["__pos__"].isna().all():
        # nearest match within ±5 minutes
        exact = pd.merge_asof(
            idx[[key_a]].sort_values(key_a),
            feats[[key_b,"__pos__"]].sort_values(key_b),
            left_on=key_a, right_on=key_b,
            direction="nearest", tolerance=pd.Timedelta("5min")
        )

    # drop any starts we couldn't align
    exact = exact.dropna(subset=["__pos__"]).reset_index(drop=True)
    starts = exact["__pos__"].astype(int).values

    # generate windows
    y = feats[target_col].values
    rows = []
    for s in starts:
        end = s + horizon
        if end <= len(y):
            rows.append(y[s:end])
    if not rows:
        raise ValueError("No aligned windows found (check time tolerance or data coverage).")

    Y = np.vstack(rows)  # (N, horizon)
    return Y


In [13]:
# --- CELL 6: METRICS (overall + per-horizon + t+48) with op_24h_latest ---
import pandas as pd, numpy as np, os

rows_overall, rows_h, rows_t48 = [], [], []

def stack_preds(path):
    return pd.read_csv(path).values

def add_model_metrics(cohort_name, label, Yt, Yp):
    # overall
    m = compute_metrics(Yt.ravel(), Yp.ravel());
    m.update({"cohort": cohort_name, "model": label})
    rows_overall.append(m)
    # per-horizon + t+48 capture
    for h in range(HORIZON):
        mh = compute_metrics(Yt[:,h], Yp[:,h]);
        mh.update({"cohort": cohort_name, "model": label, "horizon": h+1})
        rows_h.append(mh)
        if h+1 == 48:
            rows_t48.append({
                "cohort": cohort_name,
                "model": label,
                "MAE": mh["MAE"],
                "RMSE": mh["RMSE"],
                "MAPE": mh["MAPE"],
                "Bias": mh["Bias"]
            })

# compute
for name, cohort_path in COHORT.items():
    # Truths (N,48)
    Y_true = truths_from_features_for_cohort(cohort_path, features_df, target_col)  # (N,48)
    N = len(Y_true)

    # Models (LGBM, EVL if exists)
    models_to_eval = {
        "LightGBM": stack_preds(f"{PRED_DIR}/predictions__JEFF__LGBM__{name}.csv")[:N]
    }
    evl_path = f"{PRED_DIR}/predictions__JEFF__EVL__{name}.csv"
    if os.path.exists(evl_path):
        models_to_eval["EVL"] = stack_preds(evl_path)[:N]

    # Operator (Maria’s request)
    df_coh = pd.read_csv(cohort_path)
    op_col = pick_operator_col(df_coh)
    Y_true_eval = Y_true
    Y_op = None
    if op_col is not None:
        op_vec = df_coh[op_col].values
        M = len(op_vec)//HORIZON
        Y_op = op_vec[:M*HORIZON].reshape(M, HORIZON)
        K = min(N, M)
        Y_true_eval = Y_true[:K]
        Y_op = Y_op[:K]

    # Evaluate ML models
    for label, Yhat in models_to_eval.items():
        K2 = min(len(Yhat), len(Y_true_eval))
        add_model_metrics(name, label, Y_true_eval[:K2], Yhat[:K2])

    # Evaluate Operator (if present)
    if Y_op is not None:
        add_model_metrics(name, "Operator(op_24h_latest)", Y_true_eval, Y_op)

# Make dataframes
df_overall  = pd.DataFrame(rows_overall)
df_horizon  = pd.DataFrame(rows_h)
df_t48      = pd.DataFrame(rows_t48)

# Save CSVs
overall_out = f"{METRICS_DIR}/overall_metrics__cohort3.csv"
horizon_out = f"{METRICS_DIR}/horizon_metrics__cohort3.csv"
t48_out     = f"{METRICS_DIR}/t48_summary__cohort3.csv"

df_overall.to_csv(overall_out, index=False)
df_horizon.to_csv(horizon_out, index=False)
df_t48.to_csv(t48_out, index=False)

print("Wrote:", overall_out)
print("Wrote:", horizon_out)
print("Wrote:", t48_out)

# Quick peek
display(df_overall.sort_values(["cohort","model"]).round(3).head(10))
display(df_t48.sort_values(["cohort","MAE"]).round(3))


Wrote: /content/drive/MyDrive/JEFF UNI/week 6/Collab/Cohort3-Metrics/overall_metrics__cohort3.csv
Wrote: /content/drive/MyDrive/JEFF UNI/week 6/Collab/Cohort3-Metrics/horizon_metrics__cohort3.csv
Wrote: /content/drive/MyDrive/JEFF UNI/week 6/Collab/Cohort3-Metrics/t48_summary__cohort3.csv


,MAE,RMSE,MAPE,Bias,cohort,model
2,1434.392,1703.532,17.251,-157.900,hotday_all,LightGBM
3,1846.164,2248.516,22.630,165.847,hotday_all,Operator(op_24h_latest)
0,1154.386,1512.839,13.275,-231.109,overall_all,LightGBM
1,1838.791,2351.929,20.130,-928.111,overall_all,Operator(op_24h_latest)


,cohort,model,MAE,RMSE,MAPE,Bias
2,hotday_all,LightGBM,1107.595,1405.309,13.846,27.206
3,hotday_all,Operator(op_24h_latest),1446.430,1719.527,17.883,-78.071
0,overall_all,LightGBM,648.486,912.883,6.960,-320.660
1,overall_all,Operator(op_24h_latest),1779.146,2267.706,18.895,-1103.650


In [14]:
# --- CELL 7: FIGURES (horizon curve, t+48 bar, overall bars, t+48 scatter) ---
import pandas as pd, numpy as np, matplotlib.pyplot as plt, os

df_overall = pd.read_csv(f"{METRICS_DIR}/overall_metrics__cohort3.csv")
df_h       = pd.read_csv(f"{METRICS_DIR}/horizon_metrics__cohort3.csv")

# 1) MAE-by-horizon curves (like Maria's operator eval horizon plots)
for cohort in df_h["cohort"].unique():
    sub = df_h[df_h["cohort"]==cohort]
    pivot = sub.pivot_table(index="horizon", columns="model", values="MAE", aggfunc="mean")
    ax = pivot.plot(figsize=(8,5), title=f"MAE by Horizon — {cohort}")
    ax.set_xlabel("Horizon (t+)")
    ax.set_ylabel("MAE (MW)")
    fig_path = f"{FIG_DIR}/FIG_{cohort}_horizon_MAE.png"
    plt.tight_layout(); plt.savefig(fig_path, dpi=180); plt.close()
    print("Wrote:", fig_path)

# 2) t+48 MAE bar
t48 = df_h[df_h["horizon"]==48].copy()
for cohort in t48["cohort"].unique():
    sub = t48[t48["cohort"]==cohort][["model","MAE"]].sort_values("MAE")
    plt.figure(figsize=(7,5))
    plt.barh(sub["model"], sub["MAE"])
    plt.title(f"t+48 MAE — {cohort}")
    plt.xlabel("MAE (MW)")
    fig_path = f"{FIG_DIR}/FIG_{cohort}_t48_MAE.png"
    plt.tight_layout(); plt.savefig(fig_path, dpi=180); plt.close()
    print("Wrote:", fig_path)

# 3) Overall metric bars (MAE, RMSE, MAPE, Bias) — 4 small panels per cohort
metrics = ["MAE","RMSE","MAPE","Bias"]
for cohort in df_overall["cohort"].unique():
    sub = df_overall[df_overall["cohort"]==cohort]
    for m in metrics:
        plt.figure(figsize=(7,5))
        order = sub.sort_values(m)[["model",m]]
        plt.barh(order["model"], order[m])
        plt.title(f"{m} — Overall — {cohort}")
        plt.xlabel(m + (" (%)" if m=="MAPE" else " (MW)"))
        fig_path = f"{FIG_DIR}/FIG_{cohort}_overall_{m}.png"
        plt.tight_layout(); plt.savefig(fig_path, dpi=180); plt.close()
        print("Wrote:", fig_path)

# 4) t+48 scatter (Truth vs Prediction) for Operator, LightGBM, EVL
def _truth_pred_pairs(name, model_label):
    # Get aligned truths the same way we did for metrics
    Y_true = truths_from_features_for_cohort(COHORT[name], features_df, target_col)  # (N,48)
    N = len(Y_true)
    if model_label == "Operator(op_24h_latest)":
        df_coh = pd.read_csv(COHORT[name])
        op_col = pick_operator_col(df_coh)
        if op_col is None:
            return None, None
        op_vec = df_coh[op_col].values
        M = len(op_vec)//HORIZON
        Y_op = op_vec[:M*HORIZON].reshape(M, HORIZON)
        K = min(N, M)
        return Y_true[:K,47], Y_op[:K,47]
    elif model_label == "LightGBM":
        P = pd.read_csv(f"{PRED_DIR}/predictions__JEFF__LGBM__{name}.csv").values[:N]
        return Y_true[:len(P),47], P[:len(P),47]
    elif model_label == "EVL":
        evl_path = f"{PRED_DIR}/predictions__JEFF__EVL__{name}.csv"
        if not os.path.exists(evl_path):
            return None, None
        P = pd.read_csv(evl_path).values[:N]
        return Y_true[:len(P),47], P[:len(P),47]
    return None, None

scatter_models = ["Operator(op_24h_latest)","LightGBM","EVL"]
for cohort in df_overall["cohort"].unique():
    plt.figure(figsize=(6,6))
    plotted_any = False
    for ml in scatter_models:
        yt, yp = _truth_pred_pairs(cohort, ml)
        if yt is None or yp is None:
            continue
        plt.scatter(yt, yp, s=8, alpha=0.5, label=ml)
        plotted_any = True
    if plotted_any:
        lims = plt.axis()
        low = min(lims[0], lims[2]); high = max(lims[1], lims[3])
        plt.plot([low, high],[low, high], linestyle="--")
        plt.title(f"t+48 Truth vs Prediction — {cohort}")
        plt.xlabel("Truth (MW)"); plt.ylabel("Prediction (MW)")
        fig_path = f"{FIG_DIR}/FIG_{cohort}_t48_scatter.png"
        plt.tight_layout(); plt.savefig(fig_path, dpi=180); plt.close()
        print("Wrote:", fig_path)


Wrote: /content/drive/MyDrive/JEFF UNI/week 6/Collab/Cohort3-Figures/FIG_overall_all_horizon_MAE.png
Wrote: /content/drive/MyDrive/JEFF UNI/week 6/Collab/Cohort3-Figures/FIG_hotday_all_horizon_MAE.png
Wrote: /content/drive/MyDrive/JEFF UNI/week 6/Collab/Cohort3-Figures/FIG_overall_all_t48_MAE.png
Wrote: /content/drive/MyDrive/JEFF UNI/week 6/Collab/Cohort3-Figures/FIG_hotday_all_t48_MAE.png
Wrote: /content/drive/MyDrive/JEFF UNI/week 6/Collab/Cohort3-Figures/FIG_overall_all_overall_MAE.png
Wrote: /content/drive/MyDrive/JEFF UNI/week 6/Collab/Cohort3-Figures/FIG_overall_all_overall_RMSE.png
Wrote: /content/drive/MyDrive/JEFF UNI/week 6/Collab/Cohort3-Figures/FIG_overall_all_overall_MAPE.png
Wrote: /content/drive/MyDrive/JEFF UNI/week 6/Collab/Cohort3-Figures/FIG_overall_all_overall_Bias.png
Wrote: /content/drive/MyDrive/JEFF UNI/week 6/Collab/Cohort3-Figures/FIG_hotday_all_overall_MAE.png
Wrote: /content/drive/MyDrive/JEFF UNI/week 6/Collab/Cohort3-Figures/FIG_hotday_all_overall_RMSE.pn

In [16]:
# --- CELL 8: REPORT TABLES (Overall & Hot-day & t+48) ---
import pandas as pd
import numpy as np
import os

# Load the metrics we just wrote
df_overall  = pd.read_csv(f"{METRICS_DIR}/overall_metrics__cohort3.csv")
df_horizon  = pd.read_csv(f"{METRICS_DIR}/horizon_metrics__cohort3.csv")

def _fmt(df):
    return (df.copy()
              .assign(**{
                  "MAE":  lambda d: d["MAE"].round(2),
                  "RMSE": lambda d: d["RMSE"].round(2),
                  "MAPE": lambda d: d["MAPE"].round(2),
                  "Bias": lambda d: d["Bias"].round(2),
              }))

def make_overall_table_for(cohort_name):
    sub = df_overall[df_overall["cohort"] == cohort_name]
    want = ["Operator(op_24h_latest)","LightGBM","EVL"]
    # keep whichever are present, preserve order above
    present = [m for m in want if m in sub["model"].unique()]
    tbl = (sub[sub["model"].isin(present)]
           .loc[:, ["model","MAE","RMSE","MAPE","Bias"]]
           .copy())
    tbl = _fmt(tbl).rename(columns={
        "model":"Model",
        "MAE":"MAE (MW)", "RMSE":"RMSE (MW)", "MAPE":"MAPE (%)", "Bias":"Bias (MW)"
    })
    return tbl

def make_t48_table_for(cohort_name):
    sub = df_horizon[(df_horizon["cohort"]==cohort_name) & (df_horizon["horizon"]==48)]
    want = ["Operator(op_24h_latest)","LightGBM","EVL"]
    present = [m for m in want if m in sub["model"].unique()]
    tbl = (sub[sub["model"].isin(present)]
           .loc[:, ["model","MAE","RMSE","MAPE","Bias"]])
    tbl = _fmt(tbl).rename(columns={
        "model":"Model",
        "MAE":"MAE (MW)", "RMSE":"RMSE (MW)", "MAPE":"MAPE (%)", "Bias":"Bias (MW)"
    })
    return tbl# ... keep everything above the same ...

overall_out = f"{METRICS_DIR}/table_overall_for_report.csv"
hotday_out  = f"{METRICS_DIR}/table_hotday_for_report.csv"
t48_out     = f"{METRICS_DIR}/table_t48_for_report.csv"

overall_tbl.to_csv(overall_out, index=False)
hotday_tbl.to_csv(hotday_out, index=False)

t48_join = pd.concat([
    t48_overall.assign(Cohort="Overall"),
    t48_hotday.assign(Cohort="Hot-day")
], ignore_index=True).loc[:, ["Cohort","Model","MAE (MW)","RMSE (MW)","MAPE (%)","Bias (MW)"]]

t48_join.to_csv(t48_out, index=False)

print("Wrote:")
print(" -", overall_out)
print(" -", hotday_out)
print(" -", t48_out)

display(overall_tbl)
display(hotday_tbl)
display(t48_join)


Wrote:
 - /content/drive/MyDrive/JEFF UNI/week 6/Collab/Cohort3-Metrics/table_overall_for_report.csv
 - /content/drive/MyDrive/JEFF UNI/week 6/Collab/Cohort3-Metrics/table_hotday_for_report.csv
 - /content/drive/MyDrive/JEFF UNI/week 6/Collab/Cohort3-Metrics/table_t48_for_report.csv


,Model,MAE (MW),RMSE (MW),MAPE (%),Bias (MW)
0,LightGBM,1154.39,1512.84,13.28,-231.11
1,Operator(op_24h_latest),1838.79,2351.93,20.13,-928.11


,Model,MAE (MW),RMSE (MW),MAPE (%),Bias (MW)
2,LightGBM,1434.39,1703.53,17.25,-157.90
3,Operator(op_24h_latest),1846.16,2248.52,22.63,165.85


,Cohort,Model,MAE (MW),RMSE (MW),MAPE (%),Bias (MW)
0,Overall,LightGBM,648.49,912.88,6.96,-320.66
1,Overall,Operator(op_24h_latest),1779.15,2267.71,18.89,-1103.65
2,Hot-day,LightGBM,1107.59,1405.31,13.85,27.21
3,Hot-day,Operator(op_24h_latest),1446.43,1719.53,17.88,-78.07


In [17]:
# --- CELL 9: Operator vs Model deltas (overall & t+48) ---
import pandas as pd
import numpy as np

df_overall  = pd.read_csv(f"{METRICS_DIR}/overall_metrics__cohort3.csv")
df_horizon  = pd.read_csv(f"{METRICS_DIR}/horizon_metrics__cohort3.csv")

def deltas_vs_operator(overall=True):
    frames = []
    for cohort in df_overall["cohort"].unique():
        if overall:
            sub = df_overall[df_overall["cohort"]==cohort].copy()
        else:
            sub = df_horizon[(df_horizon["cohort"]==cohort) & (df_horizon["horizon"]==48)].copy()
        if "Operator(op_24h_latest)" not in sub["model"].unique():
            continue
        op = sub[sub["model"]=="Operator(op_24h_latest)"][["MAE","RMSE","MAPE","Bias"]].mean()
        for model in ["LightGBM","EVL"]:
            if model not in sub["model"].unique():
                continue
            m = sub[sub["model"]==model][["MAE","RMSE","MAPE","Bias"]].mean()
            frames.append({
                "Cohort": cohort,
                "Scope": "Overall" if overall else "t+48",
                "Model": model,
                "ΔMAE (MW)":  round(m["MAE"]  - op["MAE"], 2),
                "ΔRMSE (MW)": round(m["RMSE"] - op["RMSE"], 2),
                "ΔMAPE (pp)": round(m["MAPE"] - op["MAPE"], 2),
                "ΔBias (MW)": round(m["Bias"] - op["Bias"], 2),
                "%MAE vs Op": round(100.0 * (m["MAE"]/op["MAE"] - 1.0), 2) if op["MAE"] else np.nan
            })
    return pd.DataFrame(frames)

d_overall = deltas_vs_operator(overall=True)
d_t48     = deltas_vs_operator(overall=False)

summary = pd.concat([d_overall, d_t48], ignore_index=True)
out = f"{METRICS_DIR}/operator_eval_summary.csv"
summary.to_csv(out, index=False)
print("Wrote:", out)
display(summary)


Wrote: /content/drive/MyDrive/JEFF UNI/week 6/Collab/Cohort3-Metrics/operator_eval_summary.csv


,Cohort,Scope,Model,ΔMAE (MW),ΔRMSE (MW),ΔMAPE (pp),ΔBias (MW),%MAE vs Op
0,overall_all,Overall,LightGBM,-684.40,-839.09,-6.85,697.00,-37.22
1,hotday_all,Overall,LightGBM,-411.77,-544.98,-5.38,-323.75,-22.30
2,overall_all,t+48,LightGBM,-1130.66,-1354.82,-11.94,782.99,-63.55
3,hotday_all,t+48,LightGBM,-338.84,-314.22,-4.04,105.28,-23.43


In [20]:
# --- CELL 10 (FIXED): PACKAGE FINAL ARTIFACTS ROBUSTLY ---
from pathlib import Path
import shutil, os

FINAL_DIR = Path(BASE) / "Cohort3-FINAL"
FINAL_MET = FINAL_DIR / "Cohort3-Metrics (CSV)"
FINAL_PNG = FINAL_DIR / "Cohort3-Figures (PNG)"
FINAL_PRE = FINAL_DIR / "Cohort3-Predictions"

for p in [FINAL_DIR, FINAL_MET, FINAL_PNG, FINAL_PRE]:
    p.mkdir(parents=True, exist_ok=True)

def _safe_copytree(src_dir, dst_dir):
    """Copy a folder into dst, overwriting as needed. Falls back to file-by-file if copytree fails."""
    src = Path(src_dir); dst = Path(dst_dir)
    if not src.exists():
        print(f"[WARN] Missing source: {src}")
        return 0
    try:
        shutil.copytree(src, dst, dirs_exist_ok=True)  # Python 3.8+ supports dirs_exist_ok
        return sum(1 for f in dst.rglob("*") if f.is_file())
    except Exception as e:
        print(f"[INFO] copytree fallback for {src} → {dst}: {e}")
        count = 0
        for f in src.rglob("*"):
            if f.is_file():
                rel = f.relative_to(src)
                (dst / rel).parent.mkdir(parents=True, exist_ok=True)
                shutil.copy2(str(f), str(dst / rel))
                count += 1
        return count

n_pred = _safe_copytree(PRED_DIR, FINAL_PRE)
n_met  = _safe_copytree(METRICS_DIR, FINAL_MET)
n_fig  = _safe_copytree(FIG_DIR, FINAL_PNG)

print("Packaged to:", FINAL_DIR)
print(f" - predictions: {n_pred} files")
print(f" - metrics:     {n_met} files")
print(f" - figures:     {n_fig} files")

# (optional) also make a zip for quick sharing
try:
    zip_path = str(FINAL_DIR) + ".zip"
    shutil.make_archive(str(FINAL_DIR), 'zip', str(FINAL_DIR))
    print("ZIP archive:", zip_path)
except Exception as e:
    print("[WARN] Could not create ZIP:", e)


Packaged to: /content/drive/MyDrive/JEFF UNI/week 6/Collab/Cohort3-FINAL
 - predictions: 2 files
 - metrics:     7 files
 - figures:     14 files
ZIP archive: /content/drive/MyDrive/JEFF UNI/week 6/Collab/Cohort3-FINAL.zip


In [21]:
# --- CELL 12: BUILD A SHORT ANALYSIS REPORT (MARKDOWN) ---
import pandas as pd, numpy as np, os, textwrap, datetime

overall_path = f"{METRICS_DIR}/overall_metrics__cohort3.csv"
horizon_path = f"{METRICS_DIR}/horizon_metrics__cohort3.csv"
t48_path     = f"{METRICS_DIR}/t48_summary__cohort3.csv"
op_summary   = f"{METRICS_DIR}/operator_eval_summary.csv"

df_overall = pd.read_csv(overall_path)
df_h       = pd.read_csv(horizon_path)
df_t48     = pd.read_csv(t48_path) if os.path.exists(t48_path) else None
df_delta   = pd.read_csv(op_summary) if os.path.exists(op_summary) else None

def _fmt(x):
    return "—" if pd.isna(x) else f"{x:.2f}"

def pickrow(df, cohort, model):
    s = df[(df["cohort"]==cohort) & (df["model"]==model)]
    return s.iloc[0] if len(s) else None

def line_for(cohort_label, model_label, overall=True):
    if overall:
        r = pickrow(df_overall, cohort_label, model_label)
    else:
        r = df_h[(df_h["cohort"]==cohort_label) & (df_h["model"]==model_label) & (df_h["horizon"]==48)]
        r = r.iloc[0] if len(r) else None
    if r is None:
        return f"- {model_label}: n/a"
    return f"- {model_label}: MAE { _fmt(r['MAE']) } MW, RMSE { _fmt(r['RMSE']) } MW, MAPE { _fmt(r['MAPE']) }%, Bias { _fmt(r['Bias']) } MW"

def bullets_delta(scope):
    if df_delta is None:
        return ["(delta table not found)"]
    lines = []
    sub = df_delta[df_delta["Scope"]==scope]
    for cohort in ["overall_all","hotday_all"]:
        s2 = sub[sub["Cohort"]==cohort]
        if s2.empty:
            continue
        lines.append(f"**{cohort}**")
        for m in ["LightGBM","EVL"]:
            if m not in s2["Model"].values:
                continue
            r = s2[s2["Model"]==m].iloc[0]
            lines.append(f"- {m}: ΔMAE {r['ΔMAE (MW)']:.2f} MW, ΔRMSE {r['ΔRMSE (MW)']:.2f} MW, ΔMAPE {r['ΔMAPE (pp)']:.2f} pp, ΔBias {r['ΔBias (MW)']:.2f} MW, %MAE vs Op {r['%MAE vs Op']:.2f}%")
    return lines

ts = datetime.datetime.now().strftime("%Y-%m-%d %H:%M")
md = []

md.append(f"# Cohort3 Results — Operator vs LightGBM / EVL\n_Generated {ts}_\n")
md.append("## 1) Overall metrics")
for cohort in ["overall_all","hotday_all"]:
    md.append(f"### {cohort}")
    md.append(line_for(cohort, "Operator(op_24h_latest)", overall=True))
    md.append(line_for(cohort, "LightGBM", overall=True))
    if (df_overall["model"]=="EVL").any():
        md.append(line_for(cohort, "EVL", overall=True))
    md.append("")

md.append("## 2) t+48 metrics")
for cohort in ["overall_all","hotday_all"]:
    md.append(f"### {cohort}")
    md.append(line_for(cohort, "Operator(op_24h_latest)", overall=False))
    md.append(line_for(cohort, "LightGBM", overall=False))
    if (df_h["model"]=="EVL").any():
        md.append(line_for(cohort, "EVL", overall=False))
    md.append("")

md.append("## 3) Deltas vs Operator")
md.append("### Overall")
md += bullets_delta("Overall")
md.append("\n### t+48")
md += bullets_delta("t+48")

md.append("\n## 4) Quick takeaways")
md.append("- Operator (`op_24h_latest`) is a very strong baseline; compare *%MAE vs Op* to gauge practicality.")
md.append("- EVL (average of LGBM and ALT if present) typically reduces MAE slightly vs pure LGBM; check t+48 for peak-hour behavior.")
md.append("- Inspect **Bias**: large positive bias means overprediction; negative means underprediction (watch hot-day).")
md.append("- Horizon curves (`FIG_*_horizon_MAE.png`) show where models diverge; pay attention to horizons 36–48.")
md.append("- Use scatter (`FIG_*_t48_scatter.png`) to spot systematic skew (tilt vs diagonal).")

report_path = f"{METRICS_DIR}/Cohort3_Short_Report.md"
with open(report_path, "w") as f:
    f.write("\n".join(md))

print("Wrote report:", report_path)


Wrote report: /content/drive/MyDrive/JEFF UNI/week 6/Collab/Cohort3-Metrics/Cohort3_Short_Report.md


In [23]:
print(df["model"].unique())


['LightGBM' 'Operator(op_24h_latest)']


In [22]:
# --- CELL: EVL horizon MAE plots (Operator vs EVL) ---
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Assumes these were defined earlier in your notebook:
# BASE, METRICS_DIR, FIG_DIR
# If not, uncomment and set manually:
# BASE = "/content/drive/MyDrive/JEFF UNI/week 6/Collab"
# METRICS_DIR = f"{BASE}/Cohort3-Metrics"
# FIG_DIR = f"{BASE}/Cohort3-Figures"

metrics_csv = f"{METRICS_DIR}/horizon_metrics__cohort3.csv"
assert os.path.exists(metrics_csv), f"Missing: {metrics_csv}"

df = pd.read_csv(metrics_csv)

want_models = ["Operator(op_24h_latest)", "JEFF__EVL"]
want_cohorts = ["overall_all", "hotday_all"]

sub = (df[(df["model"].isin(want_models)) & (df["cohort"].isin(want_cohorts))]
       .copy())

# Helper to plot one cohort
def plot_horizon_mae(cohort, outfile):
    d = sub[sub["cohort"]==cohort]
    if d.empty:
        raise ValueError(f"No rows for cohort={cohort} in {metrics_csv}")

    # Expect columns: horizon, MAE, model, cohort (from previous pipeline)
    # If horizon column is missing but you have H1..H48 MAEs, adapt here.
    assert {"horizon","MAE","model"}.issubset(d.columns), \
        "Expected columns 'horizon', 'MAE', 'model' not found."

    plt.figure(figsize=(10,6))
    for m in want_models:
        dm = d[d["model"]==m].sort_values("horizon")
        plt.plot(dm["horizon"], dm["MAE"], label=m)  # default colors per policy

    plt.title(f"MAE by Horizon — {cohort}")
    plt.xlabel("Horizon (t+)")
    plt.ylabel("MAE (MW)")
    plt.legend(title="model")
    plt.tight_layout()

    os.makedirs(FIG_DIR, exist_ok=True)
    plt.savefig(outfile, dpi=200)
    plt.close()
    print("Wrote:", outfile)

out_overall = f"{FIG_DIR}/FIG_{want_cohorts[0]}_horizon_MAE_EVL.png"
out_hotday  = f"{FIG_DIR}/FIG_{want_cohorts[1]}_horizon_MAE_EVL.png"

plot_horizon_mae("overall_all", out_overall)
plot_horizon_mae("hotday_all",  out_hotday)

print("✅ EVL horizon plots ready for Figure 15a (overall) and Figure 15b (hot-day).")


Wrote: /content/drive/MyDrive/JEFF UNI/week 6/Collab/Cohort3-Figures/FIG_overall_all_horizon_MAE_EVL.png
Wrote: /content/drive/MyDrive/JEFF UNI/week 6/Collab/Cohort3-Figures/FIG_hotday_all_horizon_MAE_EVL.png
✅ EVL horizon plots ready for Figure 15a (overall) and Figure 15b (hot-day).


In [24]:
# --- ADD EVL TO HORIZON METRICS + PLOT (Fig 15a/15b) ---
import os, shutil
import numpy as np, pandas as pd
import matplotlib.pyplot as plt

# If not already defined earlier, set these:
# BASE = "/content/drive/MyDrive/JEFF UNI/week 6/Collab"
# FEATURES_CSV = f"{BASE}/Collab/nsw_demand_features.csv"    # adjust if different
# COHORT = {
#   "overall_all": f"{BASE}/Collab/3cohort_overall_all.csv",
#   "hotday_all":  f"{BASE}/Collab/3cohort_hotday_all.csv",
# }
# FIG_DIR = f"{BASE}/Cohort3-Figures"
# METRICS_DIR = f"{BASE}/Cohort3-Metrics"
# PRED_DIR = f"{BASE}/Cohort3-Predictions"

metrics_csv = f"{METRICS_DIR}/horizon_metrics__cohort3.csv"

# ---- helpers (use your earlier ones if already defined) ----
def pick_target_col(df):
    for c in ["demand_MW","demand","target","y","load_MW","load"]:
        if c in df.columns: return c
    raise ValueError("Could not find target column in features.")

def detect_key_column(df_a, df_b):
    # choose the first datetime-like or object column name shared between A and B
    common = [c for c in df_a.columns if c in df_b.columns]
    for c in common:
        if np.issubdtype(df_a[c].dtype, np.datetime64) or df_a[c].dtype==object:
            return c
    raise ValueError("Align key not found between cohort and features.")

def coerce_datetime(s):
    return pd.to_datetime(s, errors="coerce", utc=True).tz_convert(None)

def truths_from_features_for_cohort(cohort_csv, features_df, target_col):
    co = pd.read_csv(cohort_csv)
    key = detect_key_column(co, features_df)
    co[key] = coerce_datetime(co[key])
    ff = features_df.copy()
    ff[key] = coerce_datetime(ff[key])
    # keep only rows that exist in cohort order
    merged = co[[key]].merge(ff[[key, target_col]], on=key, how="left")
    y = merged[target_col].to_numpy()
    # expect N*48 rows? If features are already at horizon granularity skip reshape
    # For our pipeline we used shape (N,48); infer N by multiple of 48
    assert y.size % 48 == 0, "Truth vector is not multiple of 48; check inputs."
    return y.reshape(-1, 48)

# ---- load data needed ----
features_df = pd.read_csv(FEATURES_CSV)
target_col = pick_target_col(features_df)

def load_evl_preds(cohort):
    # try common filenames
    cand = [
        f"{PRED_DIR}/predictions__JEFF__EVL__{cohort}.csv",
        f"{PRED_DIR}/predictions__EVL__{cohort}.csv",
        f"{PRED_DIR}/EVL__{cohort}.csv",
    ]
    for p in cand:
        if os.path.exists(p):
            return pd.read_csv(p, header=None).to_numpy()
    raise FileNotFoundError(f"EVL predictions not found for {cohort} in {PRED_DIR}")

rows = []
for cohort in ["overall_all","hotday_all"]:
    Y_true = truths_from_features_for_cohort(COHORT[cohort], features_df, target_col)  # (N,48)
    Y_pred = load_evl_preds(cohort)                                                     # (N,48)
    assert Y_true.shape == Y_pred.shape, f"Shape mismatch for {cohort}: {Y_true.shape} vs {Y_pred.shape}"
    mae_by_h = np.abs(Y_pred - Y_true).mean(axis=0)                                     # (48,)
    for h, v in enumerate(mae_by_h, start=1):
        rows.append({"cohort": cohort, "model": "EVL", "horizon": h, "MAE": float(v)})

evl_df = pd.DataFrame(rows, columns=["cohort","model","horizon","MAE"])

# ---- append to horizon_metrics and save ----
os.makedirs(METRICS_DIR, exist_ok=True)
if os.path.exists(metrics_csv):
    shutil.copy(metrics_csv, metrics_csv.replace(".csv","__backup_before_evl.csv"))
    base_df = pd.read_csv(metrics_csv)
    # drop any old EVL rows to avoid duplicates
    base_df = base_df[~((base_df["model"]=="EVL") & (base_df["cohort"].isin(["overall_all","hotday_all"])))]
    out_df = pd.concat([base_df, evl_df], ignore_index=True)
else:
    out_df = evl_df
out_df.to_csv(metrics_csv, index=False)
print("Updated metrics file:", metrics_csv)

# ---- optional: make Figure 15a/15b (Operator vs EVL) ----
def plot_horizon_mae(df, cohort, outfile):
    sub = df[df["cohort"]==cohort]
    have = sorted(sub["model"].unique().tolist())
    print(f"{cohort} contains models:", have)
    plt.figure(figsize=(10,6))
    for m in ["Operator(op_24h_latest)", "EVL"]:
        if m in have:
            d = sub[sub["model"]==m].sort_values("horizon")
            plt.plot(d["horizon"], d["MAE"], label=m)  # default colors
    plt.title(f"MAE by Horizon — {cohort}")
    plt.xlabel("Horizon (t+)")
    plt.ylabel("MAE (MW)")
    plt.legend(title="model")
    plt.tight_layout()
    os.makedirs(FIG_DIR, exist_ok=True)
    plt.savefig(outfile, dpi=200); plt.close()
    print("Wrote:", outfile)

plot_horizon_mae(out_df, "overall_all", f"{FIG_DIR}/FIG_overall_all_horizon_MAE_EVL.png")
plot_horizon_mae(out_df, "hotday_all",  f"{FIG_DIR}/FIG_hotday_all_horizon_MAE_EVL.png")
print("✅ Use these for Figure 15a/15b.")


ValueError: Align key not found between cohort and features.

In [26]:
# EVL vs Operator vs LGBM visualisation
import pandas as pd
import matplotlib.pyplot as plt
import os

metrics_csv = f"{METRICS_DIR}/horizon_metrics__cohort3.csv"
df = pd.read_csv(metrics_csv)

# Choose cohorts
for cohort in ["overall_all","hotday_all"]:
    sub = df[df["cohort"]==cohort]
    have = sorted(sub["model"].unique())
    print(f"{cohort} has:", have)

    # ---- Line plot: MAE by horizon ----
    plt.figure(figsize=(10,6))
    for m in ["Operator(op_24h_latest)","LightGBM","EVL"]:
        if m in have:
            d = sub[sub["model"]==m].sort_values("horizon")
            plt.plot(d["horizon"], d["MAE"], label=m, linewidth=2)
    plt.title(f"MAE by Horizon — {cohort}")
    plt.xlabel("Horizon (t+)")
    plt.ylabel("MAE (MW)")
    plt.legend()
    plt.tight_layout()
    plt.savefig(f"{FIG_DIR}/FIG_{cohort}_horizon_MAE_COMPARE.png", dpi=200)
    plt.close()

    # ---- Bar plot: MAE at t+48 ----
    plt.figure(figsize=(6,5))
    sub48 = sub[sub["horizon"]==48]
    plt.bar(sub48["model"], sub48["MAE"], color=["orange","blue","green"][:len(sub48)])
    plt.title(f"t+48 MAE — {cohort}")
    plt.ylabel("MAE (MW)")
    plt.xticks(rotation=20)
    plt.tight_layout()
    plt.savefig(f"{FIG_DIR}/FIG_{cohort}_t48_MAE_COMPARE.png", dpi=200)
    plt.close()


overall_all has: ['LightGBM', 'Operator(op_24h_latest)']
hotday_all has: ['LightGBM', 'Operator(op_24h_latest)']


In [28]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# ==== Inputs ====
BASE = "/content/drive/MyDrive/JEFF UNI/week 6/Collab"
METRICS_DIR = f"{BASE}/Cohort3-Metrics"
PRED_DIR = f"{BASE}/Cohort3-Predictions"
FEATURES_CSV = f"{BASE}/Collab/nsw_demand_features.csv"
COHORT = {
    "overall_all": f"{BASE}/Collab/3cohort_overall_all.csv",
    "hotday_all":  f"{BASE}/Collab/3cohort_hotday_all.csv",
}
FIG_DIR = f"{BASE}/Cohort3-Figures"
os.makedirs(FIG_DIR, exist_ok=True)

# ==== Helper for truth ====
def coerce_datetime(s): return pd.to_datetime(s, errors="coerce", utc=True).tz_convert(None)

def truths_from_features_for_cohort(cohort_csv, features_df, target_col="demand_MW"):
    co = pd.read_csv(cohort_csv)
    co["date"] = coerce_datetime(co.iloc[:,0])  # assume first col is datetime
    ff = features_df.copy()
    ff["date"] = coerce_datetime(ff["date"])
    merged = co[["date"]].merge(ff[["date", target_col]], on="date", how="left")
    y = merged[target_col].to_numpy()
    return y.reshape(-1, 48)  # (N,48)

features_df = pd.read_csv(FEATURES_CSV)

# ==== A) Bar comparison at t+48 ====
t48_csv = os.path.join(METRICS_DIR, "t48_summary__cohort3.csv")
df = pd.read_csv(t48_csv)
cohorts = ["overall_all","hotday_all"]
models  = ["Operator(op_24h_latest)","LightGBM","EVL"]

for metric in ["MAE","Bias"]:
    pivot = df[df["cohort"].isin(cohorts) & df["model"].isin(models)] \
              .pivot(index="cohort", columns="model", values=metric)
    pivot = pivot.reindex(index=cohorts, columns=models)
    pivot.plot(kind="bar", figsize=(8,5))
    plt.ylabel(metric + " (MW)")
    plt.title(f"{metric} at t+48 by cohort")
    plt.tight_layout()
    plt.savefig(os.path.join(FIG_DIR, f"FIG_t48_{metric}_compare.png"), dpi=200)
    plt.close()

# ==== B) Scatterplots (Pred vs True at t+48) ====
def scatterplot(pred_csv, cohort_csv, cohort_name, model_tag):
    preds = pd.read_csv(pred_csv).values.flatten()
    y_true = truths_from_features_for_cohort(cohort_csv, features_df)
    y_true = y_true[:, -1]  # take t+48
    plt.figure(figsize=(5,5))
    plt.scatter(y_true, preds, alpha=0.4, s=10)
    lims = [min(y_true.min(), preds.min()), max(y_true.max(), preds.max())]
    plt.plot(lims, lims, "r--")
    plt.xlabel("True Load (MW)")
    plt.ylabel("Predicted Load (MW)")
    plt.title(f"{model_tag} Pred vs True at t+48 ({cohort_name})")
    plt.tight_layout()
    plt.savefig(os.path.join(FIG_DIR, f"FIG_{model_tag}_{cohort_name}_scatter.png"), dpi=200)
    plt.close()

for model_tag in ["LightGBM","EVL"]:
    for c in cohorts:
        scatterplot(os.path.join(PRED_DIR, f"predictions__JEFF__{model_tag}__{c}.csv"),
                    COHORT[c], c, model_tag)

# ==== C) Line plots (MAE by horizon) ====
mae_csv = os.path.join(METRICS_DIR, "horizon_summary__cohort3.csv")
dfh = pd.read_csv(mae_csv)

for c in cohorts:
    subset = dfh[(dfh["cohort"]==c) & (dfh["model"].isin(models))]
    plt.figure(figsize=(8,5))
    for m in models:
        part = subset[subset["model"]==m]
        plt.plot(part["horizon"], part["MAE"], label=m)
    plt.xlabel("Horizon (t+)")
    plt.ylabel("MAE (MW)")
    plt.title(f"MAE by Horizon — {c}")
    plt.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(FIG_DIR, f"FIG_{c}_MAE_by_horizon.png"), dpi=200)
    plt.close()


FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/JEFF UNI/week 6/Collab/Collab/nsw_demand_features.csv'

In [29]:
# --- PRESENTATION BAR CHARTS: t+48 MAE (and Bias) for overall_24match & hotday_24match ---
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# If these exist in your notebook, this will use them; else it sets safe defaults:
try:
    METRICS_DIR
except NameError:
    BASE = "/content/drive/MyDrive/JEFF UNI/week 6/Collab"
    METRICS_DIR = f"{BASE}/Cohort3-Metrics"
    FIG_DIR = f"{BASE}/Cohort3-Figures"
os.makedirs(METRICS_DIR, exist_ok=True)
os.makedirs(FIG_DIR, exist_ok=True)

t48_csv = f"{METRICS_DIR}/t48_summary__cohort3.csv"
df = pd.read_csv(t48_csv)

# Keep only cohorts and models we care about
cohorts = ["overall_all", "hotday_all"]
models  = ["Operator(op_24h_latest)", "LightGBM", "EVL"]
present_names = {
    "overall_all":  "Overall (24-match)",
    "hotday_all":   "Hot-day (24-match)",
    "Operator(op_24h_latest)": "Operator",
    "LightGBM": "LightGBM",
    "EVL": "EVL",
}

def make_grouped_bar(metric, fname_out):
    sub = df[df["cohort"].isin(cohorts) & df["model"].isin(models)].copy()
    if sub.empty:
        raise ValueError(f"No rows found for t+48 {metric}. Check {t48_csv}.")
    # Rename for display
    sub["cohort_disp"] = sub["cohort"].map(present_names)
    sub["model_disp"]  = sub["model"].map(present_names)
    piv = sub.pivot(index="cohort_disp", columns="model_disp", values=metric)
    # Keep columns in preferred order (drop missing gracefully)
    col_order = [present_names[m] for m in models if present_names[m] in piv.columns]
    piv = piv[col_order].reindex([present_names[c] for c in cohorts])

    # Plot
    plt.figure(figsize=(8.5, 5.2))
    x = np.arange(len(piv.index))
    n = len(piv.columns)
    width = 0.22 if n==3 else 0.28
    for i, col in enumerate(piv.columns):
        plt.bar(x + (i - (n-1)/2)*width, piv[col].values, width, label=col)

    # Value labels
    for i, col in enumerate(piv.columns):
        ys = piv[col].values
        xs = x + (i - (n-1)/2)*width
        for xi, yi in zip(xs, ys):
            if pd.notna(yi):
                plt.text(xi, yi, f"{yi:.0f}", ha="center", va="bottom", fontsize=9)

    plt.xticks(x, piv.index)
    unit = " (MW)" if metric in ["MAE","Bias","RMSE"] else ""
    plt.ylabel(f"{metric}{unit}")
    title_metric = "MAE" if metric.upper()=="MAE" else metric
    plt.title(f"{title_metric} at t+48 — Operator vs LightGBM vs EVL")
    plt.legend()
    plt.tight_layout()
    out_path = os.path.join(FIG_DIR, fname_out)
    plt.savefig(out_path, dpi=220)
    plt.close()
    print("Saved:", out_path)

# Main chart for the slide (MAE)
make_grouped_bar("MAE", "PRESO_t48_MAE_COMPARE.png")

# Optional: Bias companion chart (uncomment if you want it too)
# make_grouped_bar("Bias", "PRESO_t48_Bias_COMPARE.png")


Saved: /content/drive/MyDrive/JEFF UNI/week 6/Collab/Cohort3-Figures/PRESO_t48_MAE_COMPARE.png


In [43]:
# ===== COHORT3 FULL PIPELINE (Operator + LGBM + EVL) with MultiOutputRegressor =====
# - Operator(op_24h_latest) eval (aligned, drop NaNs, trim to 48-multiple)
# - Training allowed (ALLOW_FUTURE_TRAIN=True) per team request
# - LightGBM wrapped in MultiOutputRegressor for multi-horizon blocks
# - Outputs: predictions, t+48 & horizon metrics, report tables, and figures
# ================================================================================

!pip -q install lightgbm==4.3.0 scikit-learn==1.4.2

import os, numpy as np, pandas as pd, matplotlib.pyplot as plt
from lightgbm import LGBMRegressor
from sklearn.multioutput import MultiOutputRegressor

# ----------------- CONFIG -----------------
BASE = "/content/drive/MyDrive/JEFF UNI/week 6/Collab"  # <-- EDIT if needed

FEATURES_CSV = f"{BASE}/nsw_demand_features.csv"
COHORT = {
    "overall_all": f"{BASE}/3cohort_overall_all.csv",
    "hotday_all":  f"{BASE}/3cohort_hotday_all.csv",
}
PRED_DIR    = f"{BASE}/Cohort3-Predictions"
METRICS_DIR = f"{BASE}/Cohort3-Metrics"
FIG_DIR     = f"{BASE}/Cohort3-Figures"
os.makedirs(PRED_DIR, exist_ok=True)
os.makedirs(METRICS_DIR, exist_ok=True)
os.makedirs(FIG_DIR, exist_ok=True)

ALLOW_FUTURE_TRAIN = True  # <-- you asked to allow training

# --------------- HELPERS ------------------
def _coerce_dt(s):
    s = pd.to_datetime(s, errors="coerce", utc=True)
    try:
        if hasattr(s, "dt") and s.dt.tz is not None:
            return s.dt.tz_convert(None)
    except Exception:
        pass
    return s

def _pick_time_key(df):
    for c in ["date","timestamp","Datetime","datetime","ts","Time","time"]:
        if c in df.columns: return c
    return df.columns[0]

def _pick_target_col(df):
    for c in ["demand_MW","demand","target","y","load_MW","load"]:
        if c in df.columns: return c
    raise ValueError("Target column not found in features CSV.")

def _floor_to_48_len(n): return (n // 48) * 48
def _mae(a,b): return float(np.mean(np.abs(a-b)))
def _bias(a,b): return float(np.mean(a-b))

# --------------- LOAD FEATURES ---------------
feat = pd.read_csv(FEATURES_CSV)
tkey = _pick_time_key(feat)
tcol = _pick_target_col(feat)
feat[tkey] = _coerce_dt(feat[tkey])

# Supervised targets Y+1..Y+48 (drop tail with any NaNs)
sup = feat[[tkey, tcol]].copy().sort_values(tkey).reset_index(drop=True)
for h in range(1, 49):
    sup[f"{tcol}_tplus{h}"] = sup[tcol].shift(-h)
valid = ~sup[[f"{tcol}_tplus{h}" for h in range(1,49)]].isna().any(axis=1)
sup = sup.loc[valid].reset_index(drop=True)

# Feature matrix (numeric engineered features only), aligned to sup via time
Xall = feat.drop(columns=[c for c in [tkey, tcol] if c in feat.columns], errors="ignore")
num_cols = [c for c in Xall.columns if pd.api.types.is_numeric_dtype(Xall[c])]
Xall = Xall[num_cols].loc[valid].reset_index(drop=True)
sup_times = feat.loc[valid, tkey].reset_index(drop=True)

# --------------- OPERATOR EVALUATION (op_24h_latest; aligned & trimmed) ---------------
t48_op_rows, h_op_rows = [], []

for name, path in COHORT.items():
    co = pd.read_csv(path)
    ckey = tkey if tkey in co.columns else co.columns[0]
    co[ckey] = _coerce_dt(co[ckey])

    op_col = None
    for cand in ["op_24h_latest", "op_24h", "op_latest"]:
        if cand in co.columns:
            op_col = cand; break
    if op_col is None:
        raise ValueError(f"{name}: expected op_24h_latest column not found. Example cols: {list(co.columns)[:12]}")

    merged = co[[ckey, op_col]].merge(feat[[tkey, tcol]], left_on=ckey, right_on=tkey, how="left")
    merged = merged.dropna(subset=[op_col, tcol]).reset_index(drop=True)

    n48 = _floor_to_48_len(len(merged))
    if n48 == 0:
        raise ValueError(f"{name}: <48 aligned rows after dropping NaNs.")
    if len(merged) > n48:
        print(f"⚠️  {name}: dropped {len(merged)-n48} row(s) to form complete 48-step blocks (operator).")
    merged = merged.iloc[:n48].reset_index(drop=True)

    Y  = merged[tcol].to_numpy().reshape(-1, 48)
    OP = merged[op_col].to_numpy().reshape(-1, 48)

    t48_op_rows.append({"cohort": name, "model": "Operator(op_24h_latest)",
                        "MAE": _mae(OP[:, -1], Y[:, -1]), "Bias": _bias(OP[:, -1], Y[:, -1])})
    for h in range(48):
        h_op_rows.append({"cohort": name, "model": "Operator(op_24h_latest)",
                          "horizon": h+1, "MAE": _mae(OP[:, h], Y[:, h])})

# Save/merge operator metrics
t48_base_path = os.path.join(METRICS_DIR, "t48_summary__cohort3.csv")
t48_op_df = pd.DataFrame(t48_op_rows)
if os.path.exists(t48_base_path):
    base = pd.read_csv(t48_base_path)
    base = base[~((base["model"]=="Operator(op_24h_latest)") & (base["cohort"].isin(COHORT.keys())))]
    t48_saved = pd.concat([base, t48_op_df], ignore_index=True)
else:
    t48_saved = t48_op_df
t48_saved.to_csv(t48_base_path, index=False)

h_base_path = os.path.join(METRICS_DIR, "horizon_metrics__cohort3.csv")
h_op_df = pd.DataFrame(h_op_rows)
if os.path.exists(h_base_path):
    hb = pd.read_csv(h_base_path)
    hb = hb[~((hb["model"]=="Operator(op_24h_latest)") & (hb["cohort"].isin(COHORT.keys())))]
    h_saved = pd.concat([hb, h_op_df], ignore_index=True)
else:
    h_saved = h_op_df
h_saved.to_csv(h_base_path, index=False)
print("✅ Operator(op_24h_latest) evaluated and saved.")

# --------------- TRAIN LightGBM + EVL (ALLOW_FUTURE_TRAIN=True) ---------------
# Collect cohort timestamps
co_ts_all = []
for name, path in COHORT.items():
    co = pd.read_csv(path)
    ckey = tkey if tkey in co.columns else co.columns[0]
    co_ts_all.append(_coerce_dt(co[ckey]))
co_ts_all = pd.concat(co_ts_all).dropna().astype("datetime64[ns]").sort_values().reset_index(drop=True)
earliest_co = co_ts_all.min()

if not ALLOW_FUTURE_TRAIN:
    train_mask = sup_times < earliest_co
    split_note = "STRICT (no leakage)"
else:
    cohort_set = set(co_ts_all.to_list())
    train_mask = ~sup_times.isin(cohort_set)
    split_note = "ALLOWED (may include future relative to cohort)"

X_tr = Xall.loc[train_mask].reset_index(drop=True)
sup_tr = sup.loc[train_mask].reset_index(drop=True)

print(f"Split mode: {split_note}")
print(f"Train rows: {len(sup_tr):,} | Total supervised rows: {len(sup):,}")
if len(sup_tr) == 0:
    print("⚠️  No train rows; training on ALL rows as last resort (document in report).")
    X_tr = Xall
    sup_tr = sup

# Train 4 horizon blocks × 2 seeds using MultiOutputRegressor
blocks = [(1,12), (13,24), (25,36), (37,48)]
def make_model(seed):
    return LGBMRegressor(
        n_estimators=700, learning_rate=0.03,
        num_leaves=64, max_depth=-1,
        subsample=0.8, colsample_bytree=0.8,
        reg_alpha=0.1, reg_lambda=0.1,
        random_state=seed, n_jobs=-1
    )

models_a, models_b = [], []
for a,b in blocks:
    Y_blk = sup_tr[[f"{tcol}_tplus{h}" for h in range(a,b+1)]].to_numpy()   # (N, block_size)
    mo_a = MultiOutputRegressor(make_model(42))
    mo_b = MultiOutputRegressor(make_model(137))
    mo_a.fit(X_tr, Y_blk)
    mo_b.fit(X_tr, Y_blk)
    models_a.append(mo_a)
    models_b.append(mo_b)

def predict_for_cohort(name, path):
    # Align truth like operator stage (drop NaNs, trim to 48-multiple)
    co = pd.read_csv(path)
    ckey = tkey if tkey in co.columns else co.columns[0]
    co[ckey] = _coerce_dt(co[ckey])
    merged = co[[ckey]].merge(feat[[tkey, tcol]], left_on=ckey, right_on=tkey, how="left").dropna(subset=[tcol]).reset_index(drop=True)
    n48 = _floor_to_48_len(len(merged))
    if n48 == 0: raise ValueError(f"{name}: fewer than 48 aligned rows after dropping NaNs.")
    if len(merged) > n48:
        print(f"⚠️  {name}: dropped {len(merged)-n48} row(s) to form 48-step blocks (model eval).")
    merged = merged.iloc[:n48].reset_index(drop=True)

    # Map timestamps into supervised timeline (indices for Xall rows)
    idx_map = pd.Series(range(len(sup_times)), index=sup_times.to_numpy())
    idxs = merged[ckey].map(idx_map).to_numpy()
    ok = pd.notna(idxs)
    if ok.sum() == 0: raise ValueError(f"{name}: cohort timestamps not found in supervised timeline.")
    idxs = idxs[ok].astype(int)
    Xte = Xall.iloc[idxs]

    # Predict per block with multioutput models; EVL = mean of two seeds
    parts_L, parts_E = [], []
    for bi,(a,b) in enumerate(blocks):
        p1 = np.asarray(models_a[bi].predict(Xte))  # (N, block_size)
        p2 = np.asarray(models_b[bi].predict(Xte))  # (N, block_size)
        parts_L.append(p1)
        parts_E.append(0.5*(p1+p2))
    L = np.hstack(parts_L)  # (N,48)
    E = np.hstack(parts_E)  # (N,48)

    Y = merged[tcol].to_numpy().reshape(-1,48)
    pd.DataFrame(L).to_csv(f"{PRED_DIR}/predictions__JEFF__LGBM__{name}.csv", index=False, header=False)
    pd.DataFrame(E).to_csv(f"{PRED_DIR}/predictions__JEFF__EVL__{name}.csv",  index=False, header=False)
    return Y, L, E

results = {}
for cname, cpath in COHORT.items():
    Y, L, E = predict_for_cohort(cname, cpath)
    results[cname] = {"Y":Y, "LGBM":L, "EVL":E}
    print(f"Saved predictions for {cname}: LGBM & EVL")

# --------------- METRICS & OUTPUTS ---------------
# Merge Operator t+48 from earlier, then add LGBM/EVL
t48_base = pd.read_csv(t48_base_path) if os.path.exists(t48_base_path) else pd.DataFrame(columns=["cohort","model","MAE","Bias"])
rows=[]
for n,d in results.items():
    rows += [
        {"cohort":n, "model":"LightGBM", "MAE":_mae(d["LGBM"][:,-1], d["Y"][:,-1]), "Bias":_bias(d["LGBM"][:,-1], d["Y"][:,-1])},
        {"cohort":n, "model":"EVL",      "MAE":_mae(d["EVL"][:,-1],  d["Y"][:,-1]), "Bias":_bias(d["EVL"][:,-1],  d["Y"][:,-1])},
    ]
t48_all = pd.concat([t48_base, pd.DataFrame(rows)], ignore_index=True)
t48_all_out = os.path.join(METRICS_DIR, "t48_summary__cohort3_with_evl.csv")
t48_all.to_csv(t48_all_out, index=False)

# Horizon MAE (merge operator horizons)
h_base = pd.read_csv(h_base_path) if os.path.exists(h_base_path) else pd.DataFrame(columns=["cohort","model","horizon","MAE"])
h_rows=[]
for n,d in results.items():
    for h in range(48):
        h_rows += [
            {"cohort":n, "model":"LightGBM", "horizon":h+1, "MAE":_mae(d["LGBM"][:,h], d["Y"][:,h])},
            {"cohort":n, "model":"EVL",      "horizon":h+1, "MAE":_mae(d["EVL"][:,h],  d["Y"][:,h])},
        ]
h_all = pd.concat([h_base, pd.DataFrame(h_rows)], ignore_index=True)
h_all_out = os.path.join(METRICS_DIR, "horizon_metrics__cohort3_with_evl.csv")
h_all.to_csv(h_all_out, index=False)

# Report tables
def _table(df, metric):
    sub = df[df["cohort"].isin(COHORT.keys()) & df["model"].isin(["Operator(op_24h_latest)","LightGBM","EVL"])]
    piv = sub.pivot(index="cohort", columns="model", values=metric)
    piv = piv.reindex(index=list(COHORT.keys()))
    return piv.reset_index()

_table(t48_all, "MAE").to_csv(os.path.join(METRICS_DIR, "table_t48_MAE_for_report.csv"), index=False)
_table(t48_all, "Bias").to_csv(os.path.join(METRICS_DIR, "table_t48_Bias_for_report.csv"), index=False)

# Figures
def _bar(metric, df, outpng):
    nice = {"overall_all":"Overall (24-match)", "hotday_all":"Hot-day (24-match)"}
    sub = df[df["cohort"].isin(COHORT.keys())].copy()
    sub["cohort_disp"] = sub["cohort"].map(nice)
    piv = sub.pivot(index="cohort_disp", columns="model", values=metric)
    cols = [c for c in ["Operator(op_24h_latest)","LightGBM","EVL"] if c in piv.columns]
    piv = piv[cols].reindex(["Overall (24-match)", "Hot-day (24-match)"])
    x = np.arange(len(piv.index)); n=len(piv.columns); width=0.22 if n==3 else 0.3
    plt.figure(figsize=(8.8,5.2))
    for i,col in enumerate(piv.columns):
        xs = x + (i-(n-1)/2)*width; ys = piv[col].values
        plt.bar(xs, ys, width, label=col.replace("Operator(op_24h_latest)","Operator"))
        for xi, yi in zip(xs, ys):
            if pd.notna(yi): plt.text(xi, yi, f"{yi:.0f}", ha="center", va="bottom", fontsize=9)
    plt.xticks(x, piv.index); plt.ylabel(f"{metric} (MW)")
    plt.title(f"{metric.upper()} at t+48 — Operator vs LightGBM vs EVL")
    plt.legend(); plt.tight_layout()
    outp = os.path.join(FIG_DIR, outpng); plt.savefig(outp, dpi=220); plt.close()
    print("Saved figure:", outp)

_bar("MAE",  t48_all, "PRESO_t48_MAE_COMPARE.png")
_bar("Bias", t48_all, "PRESO_t48_Bias_COMPARE.png")

def _hline(cohort, df_h, outpng):
    nice = {"overall_all":"Overall (24-match)", "hotday_all":"Hot-day (24-match)"}
    sub = df_h[df_h["cohort"]==cohort]
    plt.figure(figsize=(10,6))
    for m in ["Operator(op_24h_latest)","LightGBM","EVL"]:
        if m in sub["model"].unique():
            d = sub[sub["model"]==m].sort_values("horizon")
            plt.plot(d["horizon"], d["MAE"], label=m.replace("Operator(op_24h_latest)","Operator"))
    plt.xlabel("Horizon (t+)"); plt.ylabel("MAE (MW)")
    plt.title(f"MAE by Horizon — {nice.get(cohort, cohort)}")
    plt.legend(); plt.tight_layout()
    outp = os.path.join(FIG_DIR, outpng); plt.savefig(outp, dpi=220); plt.close()
    print("Saved figure:", outp)

_hline("overall_all", h_all, "FIG_overall_all_horizon_MAE_COMPARE.png")
_hline("hotday_all",  h_all, "FIG_hotday_all_horizon_MAE_COMPARE.png")

print("\n✅ DONE")
print("• t+48 metrics:", t48_all_out)
print("• horizon metrics:", h_all_out)
print("• report tables:", os.path.join(METRICS_DIR, "table_t48_MAE_for_report.csv"), ",",
      os.path.join(METRICS_DIR, "table_t48_Bias_for_report.csv"))
print("• predictions dir:", PRED_DIR)
print("• figures:", os.path.join(FIG_DIR, "PRESO_t48_MAE_COMPARE.png"), ",",
      os.path.join(FIG_DIR, "PRESO_t48_Bias_COMPARE.png"))
print("• horizon figs:", os.path.join(FIG_DIR, "FIG_overall_all_horizon_MAE_COMPARE.png"), ",",
      os.path.join(FIG_DIR, "FIG_hotday_all_horizon_MAE_COMPARE.png"))
print(f"Split mode: {'ALLOWED (may include future relative to cohort)' if ALLOW_FUTURE_TRAIN else 'STRICT (no leakage)'}")


✅ Operator(op_24h_latest) evaluated and saved.
Split mode: ALLOWED (may include future relative to cohort)
Train rows: 157,776 | Total supervised rows: 196,465
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.050722 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 3713
[LightGBM] [Info] Number of data points in the train set: 157776, number of used features: 28
[LightGBM] [Info] Start training from score 8188.824373
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.011570 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 3713
[LightGBM] [Info] Number of data points in the train set: 157776, number of used features: 28
[LightGBM] [Info] Start training from score 8188.837432
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing wa

InvalidIndexError: Reindexing only valid with uniquely valued Index objects

In [44]:
# === D) Predict for each cohort, compute metrics, make tables & figures ===

# Build a de-duplicated time index to avoid InvalidIndexError
time_index = (
    pd.DataFrame({"time": sup_times})
    .reset_index()
    .rename(columns={"index":"row_id"})
    .drop_duplicates(subset=["time"], keep="first")  # keep-first to ensure uniqueness
)

def predict_for_cohort(name, path):
    # Align truth like operator stage (drop NaNs, trim to 48-multiple)
    co = pd.read_csv(path)
    ckey = tkey if tkey in co.columns else co.columns[0]
    co[ckey] = _coerce_dt(co[ckey])
    merged = co[[ckey]].merge(feat[[tkey, tcol]], left_on=ckey, right_on=tkey, how="left") \
                       .dropna(subset=[tcol]).reset_index(drop=True)

    n48 = _floor_to_48_len(len(merged))
    if n48 == 0: raise ValueError(f"{name}: <48 aligned rows after dropping NaNs.")
    if len(merged) > n48:
        print(f"⚠️  {name}: dropped {len(merged)-n48} row(s) to form 48-step blocks (model eval).")
    merged = merged.iloc[:n48].reset_index(drop=True)

    # Join timestamps to unique row_id in the supervised timeline
    stamped = merged[[ckey]].merge(time_index, left_on=ckey, right_on="time", how="left")
    if stamped["row_id"].isna().any():
        missing = int(stamped["row_id"].isna().sum())
        raise ValueError(f"{name}: {missing} cohort timestamps not found in supervised timeline.")

    row_ids = stamped["row_id"].astype(int).to_numpy()
    Xte = Xall.iloc[row_ids]

    # Predict per block; EVL = mean of two seeds
    parts_L, parts_E = [], []
    for bi,(a,b) in enumerate(blocks):
        p1 = np.asarray(models_a[bi].predict(Xte))  # (N, block)
        p2 = np.asarray(models_b[bi].predict(Xte))  # (N, block)
        parts_L.append(p1)
        parts_E.append(0.5*(p1+p2))
    L = np.hstack(parts_L)  # (N,48)
    E = np.hstack(parts_E)  # (N,48)

    Y = merged[tcol].to_numpy().reshape(-1,48)

    pd.DataFrame(L).to_csv(f"{PRED_DIR}/predictions__JEFF__LGBM__{name}.csv", index=False, header=False)
    pd.DataFrame(E).to_csv(f"{PRED_DIR}/predictions__JEFF__EVL__{name}.csv",  index=False, header=False)
    return Y, L, E

results = {}
for cname, cpath in COHORT.items():
    Y, L, E = predict_for_cohort(cname, cpath)
    results[cname] = {"Y":Y, "LGBM":L, "EVL":E}
    print(f"Saved predictions for {cname}: LGBM & EVL")

# Merge Operator t+48 from cell B, then add LGBM/EVL
t48_base = pd.read_csv(f"{METRICS_DIR}/t48_summary__cohort3.csv") if os.path.exists(f"{METRICS_DIR}/t48_summary__cohort3.csv") else pd.DataFrame(columns=["cohort","model","MAE","Bias"])
rows=[]
for n,d in results.items():
    rows += [
        {"cohort":n, "model":"LightGBM", "MAE":_mae(d["LGBM"][:,-1], d["Y"][:,-1]), "Bias":_bias(d["LGBM"][:,-1], d["Y"][:,-1])},
        {"cohort":n, "model":"EVL",      "MAE":_mae(d["EVL"][:,-1],  d["Y"][:,-1]), "Bias":_bias(d["EVL"][:,-1],  d["Y"][:,-1])},
    ]
t48_all = pd.concat([t48_base, pd.DataFrame(rows)], ignore_index=True)
t48_all_out = f"{METRICS_DIR}/t48_summary__cohort3_with_evl.csv"
t48_all.to_csv(t48_all_out, index=False)

# Horizon MAE (merge operator horizons from cell B)
h_base = pd.read_csv(f"{METRICS_DIR}/horizon_metrics__cohort3.csv") if os.path.exists(f"{METRICS_DIR}/horizon_metrics__cohort3.csv") else pd.DataFrame(columns=["cohort","model","horizon","MAE"])
h_rows=[]
for n,d in results.items():
    for h in range(48):
        h_rows += [
            {"cohort":n, "model":"LightGBM", "horizon":h+1, "MAE":_mae(d["LGBM"][:,h], d["Y"][:,h])},
            {"cohort":n, "model":"EVL",      "horizon":h+1, "MAE":_mae(d["EVL"][:,h],  d["Y"][:,h])},
        ]
h_all = pd.concat([h_base, pd.DataFrame(h_rows)], ignore_index=True)
h_all_out = f"{METRICS_DIR}/horizon_metrics__cohort3_with_evl.csv"
h_all.to_csv(h_all_out, index=False)

# Report tables
def _table(df, metric):
    sub = df[df["cohort"].isin(COHORT.keys()) & df["model"].isin(["Operator(op_24h_latest)","LightGBM","EVL"])]
    piv = sub.pivot(index="cohort", columns="model", values=metric)
    piv = piv.reindex(index=list(COHORT.keys()))
    return piv.reset_index()

_table(t48_all, "MAE").to_csv(f"{METRICS_DIR}/table_t48_MAE_for_report.csv", index=False)
_table(t48_all, "Bias").to_csv(f"{METRICS_DIR}/table_t48_Bias_for_report.csv", index=False)

# Figures
def _bar(metric, df, outpng):
    nice = {"overall_all":"Overall (24-match)", "hotday_all":"Hot-day (24-match)"}
    sub = df[df["cohort"].isin(COHORT.keys())].copy()
    sub["cohort_disp"] = sub["cohort"].map(nice)
    piv = sub.pivot(index="cohort_disp", columns="model", values=metric)
    cols = [c for c in ["Operator(op_24h_latest)","LightGBM","EVL"] if c in piv.columns]
    piv = piv[cols].reindex(["Overall (24-match)", "Hot-day (24-match)"])
    x = np.arange(len(piv.index)); n=len(piv.columns); width=0.22 if n==3 else 0.3
    plt.figure(figsize=(8.8,5.2))
    for i,col in enumerate(piv.columns):
        xs = x + (i-(n-1)/2)*width; ys = piv[col].values
        plt.bar(xs, ys, width, label=col.replace("Operator(op_24h_latest)","Operator"))
        for xi, yi in zip(xs, ys):
            if pd.notna(yi): plt.text(xi, yi, f"{yi:.0f}", ha="center", va="bottom", fontsize=9)
    plt.xticks(x, piv.index); plt.ylabel(f"{metric} (MW)")
    plt.title(f"{metric.upper()} at t+48 — Operator vs LightGBM vs EVL")
    plt.legend(); plt.tight_layout()
    outp = f"{FIG_DIR}/{outpng}"; plt.savefig(outp, dpi=220); plt.close(); print("Saved figure:", outp)

_bar("MAE",  t48_all, "PRESO_t48_MAE_COMPARE.png")
_bar("Bias", t48_all, "PRESO_t48_Bias_COMPARE.png")

def _hline(cohort, df_h, outpng):
    nice = {"overall_all":"Overall (24-match)", "hotday_all":"Hot-day (24-match)"}
    sub = df_h[df_h["cohort"]==cohort]
    plt.figure(figsize=(10,6))
    for m in ["Operator(op_24h_latest)","LightGBM","EVL"]:
        if m in sub["model"].unique():
            d = sub[sub["model"]==m].sort_values("horizon")
            plt.plot(d["horizon"], d["MAE"], label=m.replace("Operator(op_24h_latest)","Operator"))
    plt.xlabel("Horizon (t+)"); plt.ylabel("MAE (MW)")
    plt.title(f"MAE by Horizon — {nice.get(cohort, cohort)}")
    plt.legend(); plt.tight_layout()
    outp = f"{FIG_DIR}/{outpng}"; plt.savefig(outp, dpi=220); plt.close(); print("Saved figure:", outp)

_hline("overall_all", h_all, "FIG_overall_all_horizon_MAE_COMPARE.png")
_hline("hotday_all",  h_all, "FIG_hotday_all_horizon_MAE_COMPARE.png")

print("\n✅ DONE")
print("• t+48 metrics:", t48_all_out)
print("• horizon metrics:", h_all_out)
print("• report tables:", f"{METRICS_DIR}/table_t48_MAE_for_report.csv , {METRICS_DIR}/table_t48_Bias_for_report.csv")
print("• predictions dir:", PRED_DIR)
print("• figures:", f"{FIG_DIR}/PRESO_t48_MAE_COMPARE.png , {FIG_DIR}/PRESO_t48_Bias_COMPARE.png")
print("• horizon figs:", f"{FIG_DIR}/FIG_overall_all_horizon_MAE_COMPARE.png , {FIG_DIR}/FIG_hotday_all_horizon_MAE_COMPARE.png")


Saved predictions for overall_all: LGBM & EVL
Saved predictions for hotday_all: LGBM & EVL


ValueError: operands could not be broadcast together with shapes (38736,) (807,) 

In [45]:
# === D) Predict for each cohort, compute metrics, make tables & figures (BLOCK-ALIGNED) ===

# Build a de-duplicated time index to avoid InvalidIndexError
time_index = (
    pd.DataFrame({"time": sup_times})
    .reset_index()
    .rename(columns={"index":"row_id"})
    .drop_duplicates(subset=["time"], keep="first")  # ensure uniqueness
)

def predict_for_cohort(name, path):
    # Align truth like operator stage (drop NaNs, trim to 48-multiple)
    co = pd.read_csv(path)
    ckey = tkey if tkey in co.columns else co.columns[0]
    co[ckey] = _coerce_dt(co[ckey])
    merged = co[[ckey]].merge(feat[[tkey, tcol]], left_on=ckey, right_on=tkey, how="left") \
                       .dropna(subset=[tcol]).reset_index(drop=True)

    n48 = _floor_to_48_len(len(merged))
    if n48 == 0: raise ValueError(f"{name}: <48 aligned rows after dropping NaNs.")
    if len(merged) > n48:
        print(f"⚠️  {name}: dropped {len(merged)-n48} row(s) to form 48-step blocks (model eval).")
    merged = merged.iloc[:n48].reset_index(drop=True)

    # Join timestamps to unique row_id in the supervised timeline
    stamped = merged[[ckey]].merge(time_index, left_on=ckey, right_on="time", how="left")
    if stamped["row_id"].isna().any():
        missing = int(stamped["row_id"].isna().sum())
        raise ValueError(f"{name}: {missing} cohort timestamps not found in supervised timeline.")

    row_ids = stamped["row_id"].astype(int).to_numpy()
    Xte = Xall.iloc[row_ids]

    # Predict per timestamp (N rows), then REDUCE to block starts every 48 rows
    parts_L, parts_E = [], []
    for bi,(a,b) in enumerate(blocks):
        p1 = np.asarray(models_a[bi].predict(Xte))  # shape (N, block_size)
        p2 = np.asarray(models_b[bi].predict(Xte))  # shape (N, block_size)
        parts_L.append(p1)
        parts_E.append(0.5*(p1+p2))
    L_full = np.hstack(parts_L)  # (N,48)
    E_full = np.hstack(parts_E)  # (N,48)

    # Keep only block starts (0, 48, 96, ...) to align with truth blocks
    n_blocks = n48 // 48
    take = np.arange(0, n_blocks*48, 48, dtype=int)
    L = L_full[take, :]        # (n_blocks, 48)
    E = E_full[take, :]        # (n_blocks, 48)
    Y = merged[tcol].to_numpy().reshape(-1,48)  # already (n_blocks, 48)

    # Save block-level predictions (matches our tables/figures)
    pd.DataFrame(L).to_csv(f"{PRED_DIR}/predictions__JEFF__LGBM__{name}.csv", index=False, header=False)
    pd.DataFrame(E).to_csv(f"{PRED_DIR}/predictions__JEFF__EVL__{name}.csv",  index=False, header=False)
    return Y, L, E

results = {}
for cname, cpath in COHORT.items():
    Y, L, E = predict_for_cohort(cname, cpath)
    results[cname] = {"Y":Y, "LGBM":L, "EVL":E}
    print(f"Saved block-level predictions for {cname}: LGBM & EVL  (shape {L.shape})")

# Merge Operator t+48 from cell B, then add LGBM/EVL
t48_base_path = f"{METRICS_DIR}/t48_summary__cohort3.csv"
t48_base = pd.read_csv(t48_base_path) if os.path.exists(t48_base_path) else pd.DataFrame(columns=["cohort","model","MAE","Bias"])
rows=[]
for n,d in results.items():
    rows += [
        {"cohort":n, "model":"LightGBM", "MAE":_mae(d["LGBM"][:,-1], d["Y"][:,-1]), "Bias":_bias(d["LGBM"][:,-1], d["Y"][:,-1])},
        {"cohort":n, "model":"EVL",      "MAE":_mae(d["EVL"][:,-1],  d["Y"][:,-1]), "Bias":_bias(d["EVL"][:,-1],  d["Y"][:,-1])},
    ]
t48_all = pd.concat([t48_base, pd.DataFrame(rows)], ignore_index=True)
t48_all_out = f"{METRICS_DIR}/t48_summary__cohort3_with_evl.csv"
t48_all.to_csv(t48_all_out, index=False)

# Horizon MAE (merge operator horizons from cell B)
h_base_path = f"{METRICS_DIR}/horizon_metrics__cohort3.csv"
h_base = pd.read_csv(h_base_path) if os.path.exists(h_base_path) else pd.DataFrame(columns=["cohort","model","horizon","MAE"])
h_rows=[]
for n,d in results.items():
    for h in range(48):
        h_rows += [
            {"cohort":n, "model":"LightGBM", "horizon":h+1, "MAE":_mae(d["LGBM"][:,h], d["Y"][:,h])},
            {"cohort":n, "model":"EVL",      "horizon":h+1, "MAE":_mae(d["EVL"][:,h],  d["Y"][:,h])},
        ]
h_all = pd.concat([h_base, pd.DataFrame(h_rows)], ignore_index=True)
h_all_out = f"{METRICS_DIR}/horizon_metrics__cohort3_with_evl.csv"
h_all.to_csv(h_all_out, index=False)

# Report tables
def _table(df, metric):
    sub = df[df["cohort"].isin(COHORT.keys()) & df["model"].isin(["Operator(op_24h_latest)","LightGBM","EVL"])]
    piv = sub.pivot(index="cohort", columns="model", values=metric)
    piv = piv.reindex(index=list(COHORT.keys()))
    return piv.reset_index()

_table(t48_all, "MAE").to_csv(f"{METRICS_DIR}/table_t48_MAE_for_report.csv", index=False)
_table(t48_all, "Bias").to_csv(f"{METRICS_DIR}/table_t48_Bias_for_report.csv", index=False)

# Figures
def _bar(metric, df, outpng):
    nice = {"overall_all":"Overall (24-match)", "hotday_all":"Hot-day (24-match)"}
    sub = df[df["cohort"].isin(COHORT.keys())].copy()
    sub["cohort_disp"] = sub["cohort"].map(nice)
    piv = sub.pivot(index="cohort_disp", columns="model", values=metric)
    cols = [c for c in ["Operator(op_24h_latest)","LightGBM","EVL"] if c in piv.columns]
    piv = piv[cols].reindex(["Overall (24-match)", "Hot-day (24-match)"])
    x = np.arange(len(piv.index)); n=len(piv.columns); width=0.22 if n==3 else 0.3
    plt.figure(figsize=(8.8,5.2))
    for i,col in enumerate(piv.columns):
        xs = x + (i-(n-1)/2)*width; ys = piv[col].values
        plt.bar(xs, ys, width, label=col.replace("Operator(op_24h_latest)","Operator"))
        for xi, yi in zip(xs, ys):
            if pd.notna(yi): plt.text(xi, yi, f"{yi:.0f}", ha="center", va="bottom", fontsize=9)
    plt.xticks(x, piv.index); plt.ylabel(f"{metric} (MW)")
    plt.title(f"{metric.upper()} at t+48 — Operator vs LightGBM vs EVL")
    plt.legend(); plt.tight_layout()
    outp = f"{FIG_DIR}/{outpng}"; plt.savefig(outp, dpi=220); plt.close(); print("Saved figure:", outp)

_bar("MAE",  t48_all, "PRESO_t48_MAE_COMPARE.png")
_bar("Bias", t48_all, "PRESO_t48_Bias_COMPARE.png")

def _hline(cohort, df_h, outpng):
    nice = {"overall_all":"Overall (24-match)", "hotday_all":"Hot-day (24-match)"}
    sub = df_h[df_h["cohort"]==cohort]
    plt.figure(figsize=(10,6))
    for m in ["Operator(op_24h_latest)","LightGBM","EVL"]:
        if m in sub["model"].unique():
            d = sub[sub["model"]==m].sort_values("horizon")
            plt.plot(d["horizon"], d["MAE"], label=m.replace("Operator(op_24h_latest)","Operator"))
    plt.xlabel("Horizon (t+)"); plt.ylabel("MAE (MW)")
    plt.title(f"MAE by Horizon — {nice.get(cohort, cohort)}")
    plt.legend(); plt.tight_layout()
    outp = f"{FIG_DIR}/{outpng}"; plt.savefig(outp, dpi=220); plt.close(); print("Saved figure:", outp)

_hline("overall_all", h_all, "FIG_overall_all_horizon_MAE_COMPARE.png")
_hline("hotday_all",  h_all, "FIG_hotday_all_horizon_MAE_COMPARE.png")

print("\n✅ DONE (block-aligned)")
print("• t+48 metrics:", t48_all_out)
print("• horizon metrics:", h_all_out)
print("• report tables:", f"{METRICS_DIR}/table_t48_MAE_for_report.csv , {METRICS_DIR}/table_t48_Bias_for_report.csv")
print("• predictions dir:", PRED_DIR)
print("• figures:", f"{FIG_DIR}/PRESO_t48_MAE_COMPARE.png , {FIG_DIR}/PRESO_t48_Bias_COMPARE.png")
print("• horizon figs:", f"{FIG_DIR}/FIG_overall_all_horizon_MAE_COMPARE.png , {FIG_DIR}/FIG_hotday_all_horizon_MAE_COMPARE.png")


Saved block-level predictions for overall_all: LGBM & EVL  (shape (807, 48))
Saved block-level predictions for hotday_all: LGBM & EVL  (shape (81, 48))


ValueError: Index contains duplicate entries, cannot reshape

In [46]:
# === A) Setup & helpers ===
!pip -q install lightgbm==4.3.0 scikit-learn==1.4.2

import os, numpy as np, pandas as pd, matplotlib.pyplot as plt
from lightgbm import LGBMRegressor
from sklearn.multioutput import MultiOutputRegressor

# ---- CONFIG (edit BASE if needed) ----
BASE = "/content/drive/MyDrive/JEFF UNI/week 6/Collab"
FEATURES_CSV = f"{BASE}/nsw_demand_features.csv"
COHORT = {
    "overall_all": f"{BASE}/3cohort_overall_all.csv",
    "hotday_all":  f"{BASE}/3cohort_hotday_all.csv",
}
PRED_DIR    = f"{BASE}/Cohort3-Predictions"
METRICS_DIR = f"{BASE}/Cohort3-Metrics"
FIG_DIR     = f"{BASE}/Cohort3-Figures"
os.makedirs(PRED_DIR, exist_ok=True)
os.makedirs(METRICS_DIR, exist_ok=True)
os.makedirs(FIG_DIR, exist_ok=True)

# Team request: allow training without extra history
ALLOW_FUTURE_TRAIN = True

# ---- helpers ----
def _coerce_dt(s):
    s = pd.to_datetime(s, errors="coerce", utc=True)
    try:
        if hasattr(s, "dt") and s.dt.tz is not None:
            return s.dt.tz_convert(None)
    except Exception:
        pass
    return s

def _pick_time_key(df):
    for c in ["date","timestamp","Datetime","datetime","ts","Time","time"]:
        if c in df.columns: return c
    return df.columns[0]

def _pick_target_col(df):
    for c in ["demand_MW","demand","target","y","load_MW","load"]:
        if c in df.columns: return c
    raise ValueError("Target column not found in features CSV.")

def _floor_to_48_len(n): return (n // 48) * 48
def _mae(a,b): return float(np.mean(np.abs(a-b)))
def _bias(a,b): return float(np.mean(a-b))


In [47]:
# === B) Operator evaluation (op_24h_latest), aligned & trimmed ===
feat = pd.read_csv(FEATURES_CSV)
tkey = _pick_time_key(feat)
tcol = _pick_target_col(feat)
feat[tkey] = _coerce_dt(feat[tkey])

t48_op_rows, h_op_rows = [], []

for name, path in COHORT.items():
    co = pd.read_csv(path)
    ckey = tkey if tkey in co.columns else co.columns[0]
    co[ckey] = _coerce_dt(co[ckey])

    # prefer op_24h_latest; fallbacks only if present
    op_col = None
    for cand in ["op_24h_latest", "op_24h", "op_latest"]:
        if cand in co.columns:
            op_col = cand; break
    if op_col is None:
        raise ValueError(f"{name}: expected op_24h_latest column not found. Example cols: {list(co.columns)[:12]}")

    # Merge operator + truth, drop rows missing either
    merged = co[[ckey, op_col]].merge(feat[[tkey, tcol]], left_on=ckey, right_on=tkey, how="left")
    merged = merged.dropna(subset=[op_col, tcol]).reset_index(drop=True)

    # Trim BOTH together to a multiple of 48
    n48 = _floor_to_48_len(len(merged))
    if n48 == 0: raise ValueError(f"{name}: <48 aligned rows after dropping NaNs.")
    if len(merged) > n48:
        print(f"⚠️  {name}: dropped {len(merged)-n48} row(s) to form complete 48-step blocks (operator).")
    merged = merged.iloc[:n48].reset_index(drop=True)

    Y  = merged[tcol].to_numpy().reshape(-1, 48)
    OP = merged[op_col].to_numpy().reshape(-1, 48)

    # t+48
    t48_op_rows.append({"cohort": name, "model":"Operator(op_24h_latest)",
                        "MAE":_mae(OP[:,-1], Y[:,-1]), "Bias":_bias(OP[:,-1], Y[:,-1])})
    # horizon MAE
    for h in range(48):
        h_op_rows.append({"cohort":name, "model":"Operator(op_24h_latest)",
                          "horizon":h+1, "MAE":_mae(OP[:,h], Y[:,h])})

# Save/merge operator metrics
t48_base_path = f"{METRICS_DIR}/t48_summary__cohort3.csv"
t48_op_df = pd.DataFrame(t48_op_rows)
if os.path.exists(t48_base_path):
    base = pd.read_csv(t48_base_path)
    base = base[~((base["model"]=="Operator(op_24h_latest)") & (base["cohort"].isin(COHORT.keys())))]
    t48_saved = pd.concat([base, t48_op_df], ignore_index=True)
else:
    t48_saved = t48_op_df
t48_saved.to_csv(t48_base_path, index=False)

h_base_path = f"{METRICS_DIR}/horizon_metrics__cohort3.csv"
h_op_df = pd.DataFrame(h_op_rows)
if os.path.exists(h_base_path):
    hb = pd.read_csv(h_base_path)
    hb = hb[~((hb["model"]=="Operator(op_24h_latest)") & (hb["cohort"].isin(COHORT.keys())))]
    h_saved = pd.concat([hb, h_op_df], ignore_index=True)
else:
    h_saved = h_op_df
h_saved.to_csv(h_base_path, index=False)

print("✅ Operator(op_24h_latest) evaluated and saved.")


✅ Operator(op_24h_latest) evaluated and saved.


In [48]:
# === C) Train LightGBM + EVL (with training allowed) ===
# Build supervised Y+1..Y+48 and features aligned to those timestamps
sup = feat[[tkey, tcol]].copy().sort_values(tkey).reset_index(drop=True)
for h in range(1, 49):
    sup[f"{tcol}_tplus{h}"] = sup[tcol].shift(-h)
valid = ~sup[[f"{tcol}_tplus{h}" for h in range(1,49)]].isna().any(axis=1)
sup = sup.loc[valid].reset_index(drop=True)

Xall = feat.drop(columns=[c for c in [tkey, tcol] if c in feat.columns], errors="ignore")
num_cols = [c for c in Xall.columns if pd.api.types.is_numeric_dtype(Xall[c])]
Xall = Xall[num_cols].loc[valid].reset_index(drop=True)
sup_times = feat.loc[valid, tkey].reset_index(drop=True)

# Collect cohort times
co_ts_all = []
for name, path in COHORT.items():
    co = pd.read_csv(path)
    ckey = tkey if tkey in co.columns else co.columns[0]
    co_ts_all.append(_coerce_dt(co[ckey]))
co_ts_all = pd.concat(co_ts_all).dropna().astype("datetime64[ns]").sort_values().reset_index(drop=True)
earliest_co = co_ts_all.min()

if not ALLOW_FUTURE_TRAIN:
    train_mask = sup_times < earliest_co
    split_note = "STRICT (no leakage)"
else:
    cohort_set = set(co_ts_all.to_list())
    train_mask = ~sup_times.isin(cohort_set)
    split_note = "ALLOWED (may include future relative to cohort)"

X_tr = Xall.loc[train_mask].reset_index(drop=True)
sup_tr = sup.loc[train_mask].reset_index(drop=True)

print(f"Split mode: {split_note}")
print(f"Train rows: {len(sup_tr):,} | Total supervised rows: {len(sup):,}")
if len(sup_tr) == 0:
    print("⚠️  No train rows; training on ALL rows as last resort (document in report).")
    X_tr = Xall
    sup_tr = sup

# Train 4 blocks × 2 seeds with MultiOutputRegressor
blocks = [(1,12), (13,24), (25,36), (37,48)]
def make_model(seed):
    return LGBMRegressor(
        n_estimators=700, learning_rate=0.03,
        num_leaves=64, max_depth=-1,
        subsample=0.8, colsample_bytree=0.8,
        reg_alpha=0.1, reg_lambda=0.1,
        random_state=seed, n_jobs=-1
    )

models_a, models_b = [], []
for a,b in blocks:
    Y_blk = sup_tr[[f"{tcol}_tplus{h}" for h in range(a,b+1)]].to_numpy()
    mo_a = MultiOutputRegressor(make_model(42))
    mo_b = MultiOutputRegressor(make_model(137))
    mo_a.fit(X_tr, Y_blk)
    mo_b.fit(X_tr, Y_blk)
    models_a.append(mo_a)
    models_b.append(mo_b)

print("✅ Training complete (LightGBM + EVL).")


Split mode: ALLOWED (may include future relative to cohort)
Train rows: 157,776 | Total supervised rows: 196,465
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.056052 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 3713
[LightGBM] [Info] Number of data points in the train set: 157776, number of used features: 28
[LightGBM] [Info] Start training from score 8188.824373
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.036214 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 3713
[LightGBM] [Info] Number of data points in the train set: 157776, number of used features: 28
[LightGBM] [Info] Start training from score 8188.837432
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.011400 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you 

In [52]:
# --- PATCH: dedupe t+48 rows, then rebuild tables & figures ---

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os

# Re-load the latest t+48 & horizon files created earlier
t48_all_path = f"{METRICS_DIR}/t48_summary__cohort3_with_evl.csv"
h_all_path   = f"{METRICS_DIR}/horizon_metrics__cohort3_with_evl.csv"

t48_all = pd.read_csv(t48_all_path)
h_all   = pd.read_csv(h_all_path)

# 1) Deduplicate (cohort, model) by taking mean of metrics
t48_all_dedup = (t48_all
                 .groupby(["cohort","model"], as_index=False)
                 .agg({"MAE":"mean","Bias":"mean"}))

# 2) Rebuild report tables
def _table(df, metric):
    sub = df[df["cohort"].isin(COHORT.keys()) & df["model"].isin(["Operator(op_24h_latest)","LightGBM","EVL"])]
    piv = sub.pivot(index="cohort", columns="model", values=metric)
    piv = piv.reindex(index=list(COHORT.keys()))
    return piv.reset_index()

_table(t48_all_dedup, "MAE").to_csv(f"{METRICS_DIR}/table_t48_MAE_for_report.csv", index=False)
_table(t48_all_dedup, "Bias").to_csv(f"{METRICS_DIR}/table_t48_Bias_for_report.csv", index=False)

# 3) Rebuild bar figures from deduped table
def _bar(metric, df, outpng):
    nice = {"overall_all":"Overall (24-match)", "hotday_all":"Hot-day (24-match)"}
    sub = df[df["cohort"].isin(COHORT.keys())].copy()
    sub["cohort_disp"] = sub["cohort"].map(nice)
    piv = sub.pivot(index="cohort_disp", columns="model", values=metric)
    cols = [c for c in ["Operator(op_24h_latest)","LightGBM","EVL"] if c in piv.columns]
    piv = piv[cols].reindex(["Overall (24-match)", "Hot-day (24-match)"])

    x = np.arange(len(piv.index)); n=len(piv.columns); width=0.22 if n==3 else 0.3
    plt.figure(figsize=(8.8,5.2))
    for i,col in enumerate(piv.columns):
        xs = x + (i-(n-1)/2)*width; ys = piv[col].values
        plt.bar(xs, ys, width, label=col.replace("Operator(op_24h_latest)","Operator"))
        for xi, yi in zip(xs, ys):
            if pd.notna(yi): plt.text(xi, yi, f"{yi:.0f}", ha="center", va="bottom", fontsize=9)
    plt.xticks(x, piv.index); plt.ylabel(f"{metric} (MW)")
    plt.title(f"{metric.upper()} at t+48 — Operator vs LightGBM vs EVL")
    plt.legend(); plt.tight_layout()
    outp = f"{FIG_DIR}/{outpng}"; plt.savefig(outp, dpi=220); plt.close(); print("Saved figure:", outp)

_bar("MAE",  t48_all_dedup, "PRESO_t48_MAE_COMPARE.png")
_bar("Bias", t48_all_dedup, "PRESO_t48_Bias_COMPARE.png")

# 4) Horizon lines don’t pivot, so no change — but we can safely re-save to be consistent
def _hline(cohort, df_h, outpng):
    nice = {"overall_all":"Overall (24-match)", "hotday_all":"Hot-day (24-match)"}
    sub = df_h[df_h["cohort"]==cohort]
    plt.figure(figsize=(10,6))
    for m in ["Operator(op_24h_latest)","LightGBM","EVL"]:
        if m in sub["model"].unique():
            d = sub[sub["model"]==m].sort_values("horizon")
            plt.plot(d["horizon"], d["MAE"], label=m.replace("Operator(op_24h_latest)","Operator"))
    plt.xlabel("Horizon (t+)"); plt.ylabel("MAE (MW)")
    plt.title(f"MAE by Horizon — {nice.get(cohort, cohort)}")
    plt.legend(); plt.tight_layout()
    outp = f"{FIG_DIR}/{outpng}"; plt.savefig(outp, dpi=220); plt.close(); print("Saved figure:", outp)

_hline("overall_all", h_all, "FIG_overall_all_horizon_MAE_COMPARE.png")
_hline("hotday_all",  h_all, "FIG_hotday_all_horizon_MAE_COMPARE.png")

print("✅ Tables & figures rebuilt from deduplicated t+48 metrics.")
print("• Tables:", f"{METRICS_DIR}/table_t48_MAE_for_report.csv , {METRICS_DIR}/table_t48_Bias_for_report.csv")
print("• Bar charts:", f"{FIG_DIR}/PRESO_t48_MAE_COMPARE.png , {FIG_DIR}/PRESO_t48_Bias_COMPARE.png")
print("• Horizon charts:", f"{FIG_DIR}/FIG_overall_all_horizon_MAE_COMPARE.png , {FIG_DIR}/FIG_hotday_all_horizon_MAE_COMPARE.png")


Saved figure: /content/drive/MyDrive/JEFF UNI/week 6/Collab/Cohort3-Figures/PRESO_t48_MAE_COMPARE.png
Saved figure: /content/drive/MyDrive/JEFF UNI/week 6/Collab/Cohort3-Figures/PRESO_t48_Bias_COMPARE.png
Saved figure: /content/drive/MyDrive/JEFF UNI/week 6/Collab/Cohort3-Figures/FIG_overall_all_horizon_MAE_COMPARE.png
Saved figure: /content/drive/MyDrive/JEFF UNI/week 6/Collab/Cohort3-Figures/FIG_hotday_all_horizon_MAE_COMPARE.png
✅ Tables & figures rebuilt from deduplicated t+48 metrics.
• Tables: /content/drive/MyDrive/JEFF UNI/week 6/Collab/Cohort3-Metrics/table_t48_MAE_for_report.csv , /content/drive/MyDrive/JEFF UNI/week 6/Collab/Cohort3-Metrics/table_t48_Bias_for_report.csv
• Bar charts: /content/drive/MyDrive/JEFF UNI/week 6/Collab/Cohort3-Figures/PRESO_t48_MAE_COMPARE.png , /content/drive/MyDrive/JEFF UNI/week 6/Collab/Cohort3-Figures/PRESO_t48_Bias_COMPARE.png
• Horizon charts: /content/drive/MyDrive/JEFF UNI/week 6/Collab/Cohort3-Figures/FIG_overall_all_horizon_MAE_COMPARE.p

In [53]:
import pandas as pd, numpy as np, os

BASE = "/content/drive/MyDrive/JEFF UNI/week 6/Collab"
METRICS_DIR = f"{BASE}/Cohort3-Metrics"

t48 = pd.read_csv(f"{METRICS_DIR}/t48_summary__cohort3_with_evl.csv")
# collapse duplicates defensively
t48 = (t48.groupby(["cohort","model"], as_index=False)
           .agg({"MAE":"mean","Bias":"mean"}))

# Pretty print
def fmt(df):
    return (df.assign(MAE=lambda d: d["MAE"].map(lambda x: f"{x:,.1f}"),
                      Bias=lambda d: d["Bias"].map(lambda x: f"{x:,.1f}"))
              .rename(columns={"cohort":"Cohort","model":"Model"}))

print("\n=== t+48 MAE & Bias (Cohort3) ===")
print(fmt(t48).to_string(index=False))

# quick pivots if you want copy-paste tables
mae_tbl = (t48.pivot(index="cohort", columns="model", values="MAE")
              .reindex(["overall_all","hotday_all"]))
bias_tbl = (t48.pivot(index="cohort", columns="model", values="Bias")
               .reindex(["overall_all","hotday_all"]))

print("\n--- MAE table (for report) ---")
print(mae_tbl.round(1).to_string())
print("\n--- Bias table (for report) ---")
print(bias_tbl.round(1).to_string())

# sanity: show where the charts/tables are saved
print("\nFiles:")
print(" - MAE table:", os.path.join(METRICS_DIR, "table_t48_MAE_for_report.csv"))
print(" - Bias table:", os.path.join(METRICS_DIR, "table_t48_Bias_for_report.csv"))
print(" - MAE bar:  Cohort3-Figures/PRESO_t48_MAE_COMPARE.png")
print(" - Bias bar: Cohort3-Figures/PRESO_t48_Bias_COMPARE.png")
print(" - Horizon (overall): Cohort3-Figures/FIG_overall_all_horizon_MAE_COMPARE.png")
print(" - Horizon (hotday):  Cohort3-Figures/FIG_hotday_all_horizon_MAE_COMPARE.png")



=== t+48 MAE & Bias (Cohort3) ===
     Cohort                   Model     MAE    Bias
 hotday_all                     EVL 1,425.0 1,397.6
 hotday_all                LightGBM 1,268.5   714.1
 hotday_all Operator(op_24h_latest)   493.6  -274.9
overall_all                     EVL   995.3   941.0
overall_all                LightGBM   821.5   310.2
overall_all Operator(op_24h_latest)   303.9  -122.3

--- MAE table (for report) ---
model           EVL  LightGBM  Operator(op_24h_latest)
cohort                                                
overall_all   995.3     821.5                    303.9
hotday_all   1425.0    1268.5                    493.6

--- Bias table (for report) ---
model           EVL  LightGBM  Operator(op_24h_latest)
cohort                                                
overall_all   941.0     310.2                   -122.3
hotday_all   1397.6     714.1                   -274.9

Files:
 - MAE table: /content/drive/MyDrive/JEFF UNI/week 6/Collab/Cohort3-Metrics/table_t48_MA

In [54]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# ---------- Metrics ----------
def mae_by_horizon(y_true: np.ndarray, y_pred: np.ndarray) -> np.ndarray:
    """
    y_true, y_pred: shape (N, 48) or (48,) flattened by rows of samples.
    Returns MAE for horizons 1..48 as array shape (48,).
    """
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    assert y_true.shape == y_pred.shape, "y_true and y_pred must have same shape"
    # If 1D of length 48, promote to (1,48)
    if y_true.ndim == 1:
        y_true = y_true.reshape(1, -1)
        y_pred = y_pred.reshape(1, -1)
    return np.mean(np.abs(y_pred - y_true), axis=0)

# ---------- Styling (match Maria’s: clean, grid, big title strip) ----------
PLOT_DIR = Path("figures_for_report")
PLOT_DIR.mkdir(exist_ok=True, parents=True)

def style_axes(ax, title):
    ax.set_title(title, fontsize=14, fontweight="bold", pad=10)
    ax.set_xlabel("Horizon (t+1 … t+48)", fontsize=11)
    ax.set_ylabel("MAE (MW)", fontsize=11)
    ax.grid(True, alpha=0.3)
    ax.set_xlim(1, 48)

def savefig(fig, filename):
    fig.tight_layout()
    out = PLOT_DIR / filename
    fig.savefig(out, dpi=200, bbox_inches="tight")
    print(f"Saved: {out.resolve()}")


In [55]:
def plot_single_model_overall_vs_hotday(mae_overall, mae_hotday, model_name: str, tag="24match"):
    """
    mae_overall, mae_hotday: arrays of length 48 for the given model.
    Produces one figure with two lines in the same style.
    """
    h = np.arange(1, 49)
    fig, ax = plt.subplots(figsize=(8.5, 5.2))
    ax.plot(h, mae_overall, lw=2.0, label="Overall", alpha=0.95)
    ax.plot(h, mae_hotday, lw=2.0, label="Hotday", alpha=0.95, linestyle="--")
    style_axes(ax, f"MAE by Horizon — {model_name} ({tag})")
    ax.legend(frameon=False)
    savefig(fig, f"MAE_by_horizon__{model_name}__overall_vs_hotday__{tag}.png")
    plt.close(fig)


In [56]:
def plot_operator_lines(mae_overall, mae_hotday, tag="24match"):
    plot_single_model_overall_vs_hotday(mae_overall, mae_hotday, "Operator", tag)


In [57]:
def plot_operator_t48_bar(mae_overall, mae_hotday, tag="24match"):
    t48_overall = float(mae_overall[47])
    t48_hotday = float(mae_hotday[47])
    fig, ax = plt.subplots(figsize=(6.2, 4.6))
    ax.bar(["Overall", "Hotday"], [t48_overall, t48_hotday])
    style_axes(ax, f"Operator Baseline — t+48 MAE ({tag})")
    ax.set_xlabel("")
    savefig(fig, f"Operator_baseline_t48__{tag}.png")
    plt.close(fig)


In [59]:
# Example if you have DataFrames called df_truth_overall, df_pred_operator_overall etc.

y_true_overall = df_truth_overall[[f"t+{i}" for i in range(1, 49)]].values
y_op_overall   = df_pred_operator_overall[[f"t+{i}" for i in range(1, 49)]].values
y_lgbm_overall = df_pred_lgbm_overall[[f"t+{i}" for i in range(1, 49)]].values
y_evl_overall  = df_pred_evl_overall[[f"t+{i}" for i in range(1, 49)]].values

y_true_hot = df_truth_hot[[f"t+{i}" for i in range(1, 49)]].values
y_op_hot   = df_pred_operator_hot[[f"t+{i}" for i in range(1, 49)]].values
y_lgbm_hot = df_pred_lgbm_hot[[f"t+{i}" for i in range(1, 49)]].values
y_evl_hot  = df_pred_evl_hot[[f"t+{i}" for i in range(1, 49)]].values


NameError: name 'df_truth_overall' is not defined

In [60]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# ---------- Metrics (vectorized across samples) ----------
def mae(y_true, y_pred, axis=0):
    return np.mean(np.abs(y_pred - y_true), axis=axis)

def rmse(y_true, y_pred, axis=0):
    return np.sqrt(np.mean((y_pred - y_true)**2, axis=axis))

def mape(y_true, y_pred, axis=0, eps=1e-9):
    # Avoid divide-by-zero using eps; y_true expected in MW
    return np.mean(np.abs((y_pred - y_true) / (np.abs(y_true) + eps)) * 100.0, axis=axis)

def bias(y_true, y_pred, axis=0):
    # sign: positive => over-forecast (pred - true)
    return np.mean((y_pred - y_true), axis=axis)

def horizon_metrics(y_true, y_pred):
    """
    Expects arrays of shape (N_samples, 48).
    Returns dict of arrays (length=48) for MAE, RMSE, MAPE, BIAS.
    """
    y_true = np.asarray(y_true); y_pred = np.asarray(y_pred)
    assert y_true.shape == y_pred.shape, "Shapes must match"
    if y_true.ndim == 1:
        y_true = y_true.reshape(1, -1); y_pred = y_pred.reshape(1, -1)
    return {
        "mae":  mae(y_true, y_pred, axis=0),
        "rmse": rmse(y_true, y_pred, axis=0),
        "mape": mape(y_true, y_pred, axis=0),
        "bias": bias(y_true, y_pred, axis=0),
    }

# Output directory
FIG_DIR = Path("figures_for_report"); FIG_DIR.mkdir(parents=True, exist_ok=True)


In [61]:
def _format_axes(ax, title=None, xlabel=None, ylabel=None):
    if title: ax.set_title(title, fontsize=13, fontweight="bold", pad=6)
    if xlabel: ax.set_xlabel(xlabel, fontsize=10)
    if ylabel: ax.set_ylabel(ylabel, fontsize=10)
    ax.grid(True, alpha=0.3)
    ax.set_xlim(1, 48)

def plot_four_panel_overall_vs_hotday(metrics_overall, metrics_hot, model_name, tag="24match"):
    """
    Two lines per panel: overall vs hotday, like your operator example.
    """
    h = np.arange(1, 49)
    fig, axs = plt.subplots(2, 2, figsize=(11, 7))
    fig.suptitle(f"{model_name} — overall vs hotday ({tag})", fontsize=15, fontweight="bold", y=0.98)

    ax = axs[0,0]
    ax.plot(h, metrics_overall["mae"],  lw=2, label="overall_all")
    ax.plot(h, metrics_hot["mae"],     lw=2, label="hotday_all", linestyle="--")
    _format_axes(ax, "MAE by Horizon (MW)", None, "MAE (MW)")
    ax.legend(frameon=False, fontsize=9)

    ax = axs[0,1]
    ax.plot(h, metrics_overall["rmse"], lw=2, label="overall_all")
    ax.plot(h, metrics_hot["rmse"],     lw=2, label="hotday_all", linestyle="--")
    _format_axes(ax, "RMSE by Horizon (MW)", None, "RMSE (MW)")

    ax = axs[1,0]
    ax.plot(h, metrics_overall["mape"], lw=2, label="overall_all")
    ax.plot(h, metrics_hot["mape"],     lw=2, label="hotday_all", linestyle="--")
    _format_axes(ax, "MAPE by Horizon (%)", "Horizon (slots ahead)", "MAPE (%)")

    ax = axs[1,1]
    ax.plot(h, metrics_overall["bias"], lw=2, label="overall_all")
    ax.plot(h, metrics_hot["bias"],     lw=2, label="hotday_all", linestyle="--")
    _format_axes(ax, "Bias by Horizon (pred − true, MW)", "Horizon (slots ahead)", "Bias (MW)")

    fig.tight_layout(rect=[0,0,1,0.95])
    out = FIG_DIR / f"{model_name.lower()}__overall_vs_hotday__fourpanel__{tag}.png"
    fig.savefig(out, dpi=200, bbox_inches="tight")
    plt.close(fig)
    print(f"Saved: {out.resolve()}")

def plot_four_panel_single(metrics, model_name, cohort_name, tag="24match"):
    """
    Single line per panel (if you want one figure per cohort too).
    """
    h = np.arange(1, 49)
    fig, axs = plt.subplots(2, 2, figsize=(11, 7))
    fig.suptitle(f"{model_name} — {cohort_name} ({tag})", fontsize=15, fontweight="bold", y=0.98)

    ax = axs[0,0]; ax.plot(h, metrics["mae"],  lw=2)
    _format_axes(ax, "MAE by Horizon (MW)", None, "MAE (MW)")
    ax = axs[0,1]; ax.plot(h, metrics["rmse"], lw=2)
    _format_axes(ax, "RMSE by Horizon (MW)", None, "RMSE (MW)")
    ax = axs[1,0]; ax.plot(h, metrics["mape"], lw=2)
    _format_axes(ax, "MAPE by Horizon (%)", "Horizon (slots ahead)", "MAPE (%)")
    ax = axs[1,1]; ax.plot(h, metrics["bias"], lw=2)
    _format_axes(ax, "Bias by Horizon (pred − true, MW)", "Horizon (slots ahead)", "Bias (MW)")

    fig.tight_layout(rect=[0,0,1,0.95])
    out = FIG_DIR / f"{model_name.lower()}__{cohort_name}__fourpanel__{tag}.png"
    fig.savefig(out, dpi=200, bbox_inches="tight")
    plt.close(fig)
    print(f"Saved: {out.resolve()}")


In [68]:
import pandas as pd

path_overall = r"/content/drive/MyDrive/JEFF UNI/week 6/Collab/3cohort_overall_all.csv"
path_hot     = r"/content/drive/MyDrive/JEFF UNI/week 6/Collab/3cohort_hotday_all.csv"

df_overall = pd.read_csv(path_overall)
df_hot     = pd.read_csv(path_hot)

print("Overall columns:", df_overall.columns[:30].tolist())
print("Hotday columns:", df_hot.columns[:30].tolist())


Overall columns: ['timestamp', 'is_hot_day', 'true', 'op_latest', 'op_24h', 'op_24h_latest']
Hotday columns: ['timestamp', 'is_hot_day', 'true', 'op_latest', 'op_24h', 'op_24h_latest']


In [70]:
import pandas as pd

METRICS_CSV = r"/content/drive/MyDrive/JEFF UNI/week 6/Collab/Cohort3-FINAL/Cohort3-Metrics (CSV)/horizon_metrics__cohort3.csv"
df = pd.read_csv(METRICS_CSV)
print(df.columns.tolist())
print(df.head(5))


['MAE', 'RMSE', 'MAPE', 'Bias', 'cohort', 'model', 'horizon']
           MAE         RMSE       MAPE        Bias       cohort     model  \
0  1798.599959  2092.491041  21.188289 -119.941166  overall_all  LightGBM   
1  1816.680646  2111.445662  21.368215 -123.917445  overall_all  LightGBM   
2  1817.593099  2118.718276  21.363873 -135.524041  overall_all  LightGBM   
3  1809.056250  2116.567826  21.228663 -163.539100  overall_all  LightGBM   
4  1819.061303  2134.365479  21.283803 -181.236911  overall_all  LightGBM   

   horizon  
0        1  
1        2  
2        3  
3        4  
4        5  


In [71]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# ---- Paths ----
METRICS_CSV = r"/content/drive/MyDrive/JEFF UNI/week 6/Collab/Cohort3-FINAL/Cohort3-Metrics (CSV)/horizon_metrics__cohort3.csv"
OUT_DIR = Path("figures_for_report"); OUT_DIR.mkdir(parents=True, exist_ok=True)

# ---- Load ----
df = pd.read_csv(METRICS_CSV)

# Normalize a bit
df["cohort_norm"] = df["cohort"].str.strip().str.lower().map(lambda s: "hotday_all" if "hot" in s else "overall_all")
df["model_norm"]  = df["model"].str.strip().str.lower().map(
    lambda s: "operator" if "oper" in s or s in {"op","operator"} else
              "lightgbm" if "lightgbm" in s or "lgbm" in s else
              "evl"      if "evl" in s or "ensemble" in s else s
)

# Helper to fetch series
def series(df_m, metric: str, cohort: str):
    sub = df_m[(df_m["cohort_norm"]==cohort)].sort_values("horizon")
    if sub.empty: return None, None
    return sub["horizon"].to_numpy(), sub[metric].to_numpy()

# 4-panel styling (matches your operator examples)
def _fmt(ax, title=None, xlabel=None, ylabel=None):
    if title:  ax.set_title(title, fontsize=13, fontweight="bold", pad=6)
    if xlabel: ax.set_xlabel(xlabel, fontsize=10)
    if ylabel: ax.set_ylabel(ylabel, fontsize=10)
    ax.grid(True, alpha=0.3)
    ax.set_xlim(1, 48)

def plot_model(model_key: str, display_name: str, tag="24match"):
    mdf = df[df["model_norm"]==model_key]
    if mdf.empty:
        print(f"[skip] Model not found in CSV: {display_name} (key={model_key})")
        return

    H_o, V_mae_o  = series(mdf, "MAE",  "overall_all")
    H_h, V_mae_h  = series(mdf, "MAE",  "hotday_all")

    H_o2, V_rmse_o = series(mdf, "RMSE", "overall_all")
    H_h2, V_rmse_h = series(mdf, "RMSE", "hotday_all")

    H_o3, V_mape_o = series(mdf, "MAPE", "overall_all")
    H_h3, V_mape_h = series(mdf, "MAPE", "hotday_all")

    H_o4, V_bias_o = series(mdf, "Bias", "overall_all")
    H_h4, V_bias_h = series(mdf, "Bias", "hotday_all")

    fig, axs = plt.subplots(2, 2, figsize=(11, 7))
    fig.suptitle(f"{display_name} — overall vs hotday ({tag})", fontsize=15, fontweight="bold", y=0.98)

    ax = axs[0,0]
    if H_o is not None: ax.plot(H_o, V_mae_o, lw=2, label="overall_all")
    if H_h is not None: ax.plot(H_h, V_mae_h, lw=2, linestyle="--", label="hotday_all")
    _fmt(ax, "MAE by Horizon (MW)", None, "MAE (MW)")
    ax.legend(frameon=False, fontsize=9)

    ax = axs[0,1]
    if H_o2 is not None: ax.plot(H_o2, V_rmse_o, lw=2, label="overall_all")
    if H_h2 is not None: ax.plot(H_h2, V_rmse_h, lw=2, linestyle="--", label="hotday_all")
    _fmt(ax, "RMSE by Horizon (MW)", None, "RMSE (MW)")

    ax = axs[1,0]
    if H_o3 is not None: ax.plot(H_o3, V_mape_o, lw=2, label="overall_all")
    if H_h3 is not None: ax.plot(H_h3, V_mape_h, lw=2, linestyle="--", label="hotday_all")
    _fmt(ax, "MAPE by Horizon (%)", "Horizon (slots ahead)", "MAPE (%)")

    ax = axs[1,1]
    if H_o4 is not None: ax.plot(H_o4, V_bias_o, lw=2, label="overall_all")
    if H_h4 is not None: ax.plot(H_h4, V_bias_h, lw=2, linestyle="--", label="hotday_all")
    _fmt(ax, "Bias by Horizon (pred − true, MW)", "Horizon (slots ahead)", "Bias (MW)")

    fig.tight_layout(rect=[0,0,1,0.95])
    out = OUT_DIR / f"{display_name.lower()}__overall_vs_hotday__fourpanel__24match.png"
    fig.savefig(out, dpi=200, bbox_inches="tight")
    plt.close(fig)
    print(f"Saved: {out.resolve()}")

# Generate Operator, LGBM, EVL
plot_model("operator", "Operator")
plot_model("lightgbm", "LGBM")
plot_model("evl", "EVL")

# (Optional) Operator baseline t+48 bar (MAE)
op = df[df["model_norm"]=="operator"].copy()
if not op.empty:
    t48_overall = op[(op["cohort_norm"]=="overall_all") & (op["horizon"]==48)]["MAE"].mean()
    t48_hot     = op[(op["cohort_norm"]=="hotday_all") & (op["horizon"]==48)]["MAE"].mean()
    fig, ax = plt.subplots(figsize=(5.5,4.4))
    ax.bar(["Overall","Hotday"], [t48_overall, t48_hot])
    ax.set_title("Operator baseline — t+48 MAE (24match)", fontsize=13, fontweight="bold")
    ax.set_ylabel("MAE (MW)")
    ax.grid(axis="y", alpha=0.3)
    out = OUT_DIR / "operator__baseline_t48_mae__24match.png"
    fig.tight_layout(); fig.savefig(out, dpi=200, bbox_inches="tight"); plt.close(fig)
    print(f"Saved: {out.resolve()}")
else:
    print("[info] No Operator rows found for t+48 bar.")


Saved: /content/figures_for_report/operator__overall_vs_hotday__fourpanel__24match.png
Saved: /content/figures_for_report/lgbm__overall_vs_hotday__fourpanel__24match.png
[skip] Model not found in CSV: EVL (key=evl)
Saved: /content/figures_for_report/operator__baseline_t48_mae__24match.png


In [72]:
# Show all unique model names in the metrics file
print(df["model"].unique())


['LightGBM' 'Operator(op_24h_latest)']


In [81]:
# --- EVL: read headerless CSVs, load truth if present, compute metrics, plot  ---
from pathlib import Path
import pandas as pd
import numpy as np

PRED_DIR = Path("/content/drive/MyDrive/JEFF UNI/week 6/Collab/Cohort3-Predictions")
evl_overall_fp = PRED_DIR / "predictions__JEFF__EVL__overall_all.csv"
evl_hot_fp     = PRED_DIR / "predictions__JEFF__EVL__hotday_all.csv"

# 1) Read EVL preds with no header -> (N,48)
evl_overall_df = pd.read_csv(evl_overall_fp, header=None)
evl_hot_df     = pd.read_csv(evl_hot_fp,     header=None)

# ensure 48 columns
assert evl_overall_df.shape[1] == 48 and evl_hot_df.shape[1] == 48, \
    f"Expected 48 columns, got {evl_overall_df.shape[1]} and {evl_hot_df.shape[1]}."

y_evl_overall = evl_overall_df.to_numpy()
y_evl_hot     = evl_hot_df.to_numpy()

# 2) Try to find truth matrices saved as CSVs (wide 48-col)
truth_overall_cands = [
    PRED_DIR / "y_true__overall_24match.csv",
    PRED_DIR / "truth__overall_24match.csv",
    PRED_DIR / "y_true__overall_all.csv",
]
truth_hot_cands = [
    PRED_DIR / "y_true__hotday_24match.csv",
    PRED_DIR / "truth__hotday_24match.csv",
    PRED_DIR / "y_true__hotday_all.csv",
]

def load_truth(cands):
    for p in cands:
        if p.exists():
            df = pd.read_csv(p)
            # accept either t+1..t+48 or any 48 columns
            cols = [f"t+{i}" for i in range(1,49)]
            if all(c in df.columns for c in cols):
                return df[cols].to_numpy(), p
            if df.shape[1] == 48:
                return df.iloc[:, :48].to_numpy(), p
    return None, None

y_true_overall, found_overall = load_truth(truth_overall_cands)
y_true_hot,     found_hot     = load_truth(truth_hot_cands)

if y_true_overall is None or y_true_hot is None:
    missing = []
    if y_true_overall is None: missing.append("overall truth (e.g. y_true__overall_24match.csv)")
    if y_true_hot is None:     missing.append("hotday truth (e.g. y_true__hotday_24match.csv)")
    raise RuntimeError(
        "EVL metrics need the truth matrices and I couldn't find them.\n"
        f"Please export and place these files in {PRED_DIR}:\n"
        "  - y_true__overall_24match.csv  (48 columns: t+1..t+48)\n"
        "  - y_true__hotday_24match.csv   (48 columns: t+1..t+48)\n"
        f"Missing: {', '.join(missing)}"
    )

print(f"Found truth files:\n - {found_overall.name}\n - {found_hot.name}")

# 3) Compute per-horizon metrics for EVL
def horizon_metrics(y_true, y_pred):
    y_true = np.asarray(y_true); y_pred = np.asarray(y_pred)
    assert y_true.shape == y_pred.shape and y_true.shape[1] == 48
    mae  = np.mean(np.abs(y_pred - y_true), axis=0)
    rmse = np.sqrt(np.mean((y_pred - y_true)**2, axis=0))
    mape = np.mean(np.abs((y_pred - y_true) / (np.abs(y_true) + 1e-9)) * 100.0, axis=0)
    bias = np.mean((y_pred - y_true), axis=0)
    return {"MAE": mae, "RMSE": rmse, "MAPE": mape, "Bias": bias}

evl_overall = horizon_metrics(y_true_overall, y_evl_overall)
evl_hot     = horizon_metrics(y_true_hot,     y_evl_hot)

# 4) Append to df and plot
def to_rows(metrics, cohort_name):
    H = np.arange(1,49)
    return pd.DataFrame({
        "MAE": metrics["MAE"],
        "RMSE": metrics["RMSE"],
        "MAPE": metrics["MAPE"],
        "Bias": metrics["Bias"],
        "cohort": cohort_name,
        "model": "EVL",
        "horizon": H
    })

evl_df = pd.concat([to_rows(evl_overall, "overall_all"),
                    to_rows(evl_hot,     "hotday_all")], ignore_index=True)
df = pd.concat([df, evl_df], ignore_index=True)

# normalize fields expected by plotter
df["cohort_norm"] = df["cohort"].astype(str).str.strip().str.lower().map(lambda s: "hotday_all" if "hot" in s else "overall_all")
df["model_norm"]  = df["model"].astype(str).str.lower().map(lambda s: "evl" if "evl" in s or "ensemble" in s else ("lightgbm" if "lightgbm" in s or "lgbm" in s else ("operator" if "oper" in s else s)))

# render EVL 4-panel
plot_model("evl", "EVL")
print("✅ EVL figure saved at:", (OUT_DIR / "evl__overall_vs_hotday__fourpanel__24match.png").resolve())


RuntimeError: EVL metrics need the truth matrices and I couldn't find them.
Please export and place these files in /content/drive/MyDrive/JEFF UNI/week 6/Collab/Cohort3-Predictions:
  - y_true__overall_24match.csv  (48 columns: t+1..t+48)
  - y_true__hotday_24match.csv   (48 columns: t+1..t+48)
Missing: overall truth (e.g. y_true__overall_24match.csv), hotday truth (e.g. y_true__hotday_24match.csv)

In [80]:
# overall_24match_truth.shape == (N, 48)
pd.DataFrame(overall_24match_truth, columns=[f"t+{i}" for i in range(1,49)]) \
  .to_csv("/content/drive/MyDrive/JEFF UNI/week 6/Collab/Cohort3-Predictions/y_true__overall_24match.csv", index=False)

# hotday_24match_truth.shape == (N, 48)
pd.DataFrame(hotday_24match_truth, columns=[f"t+{i}" for i in range(1,49)]) \
  .to_csv("/content/drive/MyDrive/JEFF UNI/week 6/Collab/Cohort3-Predictions/y_true__hotday_24match.csv", index=False)


NameError: name 'overall_24match_truth' is not defined

In [82]:
# --- EVL metrics/plot with auto truth detection (truth files or LGBM fallback) ---
from pathlib import Path
import pandas as pd, numpy as np

PRED_DIR = Path("/content/drive/MyDrive/JEFF UNI/week 6/Collab/Cohort3-Predictions")

# EVL predictions are headerless 48-col matrices
evl_overall_fp = PRED_DIR / "predictions__JEFF__EVL__overall_all.csv"
evl_hot_fp     = PRED_DIR / "predictions__JEFF__EVL__hotday_all.csv"
evl_overall_df = pd.read_csv(evl_overall_fp, header=None)
evl_hot_df     = pd.read_csv(evl_hot_fp,     header=None)
assert evl_overall_df.shape[1] == 48 and evl_hot_df.shape[1] == 48, "EVL CSVs must have 48 columns."
y_evl_overall = evl_overall_df.to_numpy()
y_evl_hot     = evl_hot_df.to_numpy()

# Try truth files first
truth_overall_cands = [
    PRED_DIR / "y_true__overall_24match.csv",
    PRED_DIR / "truth__overall_24match.csv",
    PRED_DIR / "y_true__overall_all.csv",
]
truth_hot_cands = [
    PRED_DIR / "y_true__hotday_24match.csv",
    PRED_DIR / "truth__hotday_24match.csv",
    PRED_DIR / "y_true__hotday_all.csv",
]

def load_truth_if_exists(cands):
    for p in cands:
        if p.exists():
            df = pd.read_csv(p)
            cols = [f"t+{i}" for i in range(1,49)]
            if all(c in df.columns for c in cols):
                return df[cols].to_numpy(), p.name
            if df.shape[1] == 48:
                return df.iloc[:, :48].to_numpy(), p.name
    return None, None

y_true_overall, src_overall = load_truth_if_exists(truth_overall_cands)
y_true_hot,     src_hot     = load_truth_if_exists(truth_hot_cands)

# If missing, try to derive truth from LGBM CSVs (headerless 96-col or similar)
if y_true_overall is None or y_true_hot is None:
    lgbm_overall_fp = PRED_DIR / "predictions__JEFF__LGBM__overall_all.csv"
    lgbm_hot_fp     = PRED_DIR / "predictions__JEFF__LGBM__hotday_all.csv"
    if not lgbm_overall_fp.exists() or not lgbm_hot_fp.exists():
        raise RuntimeError(
            "Missing truth files and cannot fall back to LGBM (files not found).\n"
            f"Please export:\n  {PRED_DIR/'y_true__overall_24match.csv'}\n  {PRED_DIR/'y_true__hotday_24match.csv'}"
        )

    # Read LGBM both with and without headers to handle various dumps
    lgbm_overall_h0 = pd.read_csv(lgbm_overall_fp, header=None)
    lgbm_hot_h0     = pd.read_csv(lgbm_hot_fp,     header=None)

    # Heuristic: if 96+ columns, assume first 48 are truth (common dump)
    if y_true_overall is None:
        if lgbm_overall_h0.shape[1] >= 96:
            y_true_overall = lgbm_overall_h0.iloc[:, :48].to_numpy()
            src_overall = f"{lgbm_overall_fp.name} (assumed cols 1–48 = truth)"
        else:
            raise RuntimeError(
                "Could not find overall truth. LGBM overall file does not contain 96+ columns to extract truth.\n"
                f"Please export: {PRED_DIR/'y_true__overall_24match.csv'}"
            )
    if y_true_hot is None:
        if lgbm_hot_h0.shape[1] >= 96:
            y_true_hot = lgbm_hot_h0.iloc[:, :48].to_numpy()
            src_hot = f"{lgbm_hot_fp.name} (assumed cols 1–48 = truth)"
        else:
            raise RuntimeError(
                "Could not find hotday truth. LGBM hotday file does not contain 96+ columns to extract truth.\n"
                f"Please export: {PRED_DIR/'y_true__hotday_24match.csv'}"
            )

print("Using truth sources:")
print(" - overall:", src_overall)
print(" - hotday :", src_hot)

# Sanity checks: shapes must align with EVL preds
assert y_true_overall.shape[1] == 48 and y_true_hot.shape[1] == 48, "Truth must have 48 horizons."
if y_true_overall.shape[0] != y_evl_overall.shape[0]:
    # align by trimming to the min length
    n = min(y_true_overall.shape[0], y_evl_overall.shape[0])
    y_true_overall = y_true_overall[:n]; y_evl_overall = y_evl_overall[:n]
if y_true_hot.shape[0] != y_evl_hot.shape[0]:
    n = min(y_true_hot.shape[0], y_evl_hot.shape[0])
    y_true_hot = y_true_hot[:n]; y_evl_hot = y_evl_hot[:n]

# Compute EVL metrics
def horizon_metrics(y_true, y_pred):
    y_true = np.asarray(y_true); y_pred = np.asarray(y_pred)
    mae  = np.mean(np.abs(y_pred - y_true), axis=0)
    rmse = np.sqrt(np.mean((y_pred - y_true)**2, axis=0))
    mape = np.mean(np.abs((y_pred - y_true) / (np.abs(y_true) + 1e-9))*100.0, axis=0)
    bias = np.mean((y_pred - y_true), axis=0)
    return {"MAE": mae, "RMSE": rmse, "MAPE": mape, "Bias": bias}

evl_overall = horizon_metrics(y_true_overall, y_evl_overall)
evl_hot     = horizon_metrics(y_true_hot,     y_evl_hot)

# Append EVL to df and normalize fields
def to_rows(metrics, cohort_name):
    H = np.arange(1,49)
    return pd.DataFrame({
        "MAE": metrics["MAE"],
        "RMSE": metrics["RMSE"],
        "MAPE": metrics["MAPE"],
        "Bias": metrics["Bias"],
        "cohort": cohort_name,
        "model": "EVL",
        "horizon": H
    })

df = pd.concat([df, to_rows(evl_overall, "overall_all"), to_rows(evl_hot, "hotday_all")], ignore_index=True)
df["cohort_norm"] = df["cohort"].astype(str).str.lower().map(lambda s: "hotday_all" if "hot" in s else "overall_all")
df["model_norm"]  = df["model"].astype(str).str.lower().map(lambda s: "evl" if "evl" in s or "ensemble" in s else ("lightgbm" if "lightgbm" in s or "lgbm" in s else ("operator" if "oper" in s else s)))

# Generate the EVL figure
plot_model("evl", "EVL")
print("Saved:", (OUT_DIR / "evl__overall_vs_hotday__fourpanel__24match.png").resolve())


RuntimeError: Could not find overall truth. LGBM overall file does not contain 96+ columns to extract truth.
Please export: /content/drive/MyDrive/JEFF UNI/week 6/Collab/Cohort3-Predictions/y_true__overall_24match.csv

In [85]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from pprint import pprint

METRICS_DIR = Path("/content/drive/MyDrive/JEFF UNI/week 6/Collab/Cohort3-FINAL/Cohort3-Metrics (CSV)")
OUT_DIR = Path("figures_for_report"); OUT_DIR.mkdir(exist_ok=True, parents=True)

def plot_model_from_df(df, model_key, display_name, tag="24match"):
    df = df.copy()
    df["cohort_norm"] = df["cohort"].astype(str).str.lower().map(lambda s: "hotday_all" if "hot" in s else "overall_all")
    df["model_norm"]  = df["model"].astype(str).str.lower().map(lambda s:
        "operator" if "oper" in s else ("lightgbm" if "lightgbm" in s or "lgbm" in s else ("evl" if "evl" in s or "ensemble" in s else s))
    )
    mdf = df[df["model_norm"]==model_key]
    if mdf.empty:
        return False
    def series(metric, cohort):
        sub = mdf[(mdf["cohort_norm"]==cohort)].sort_values("horizon")
        return sub["horizon"].to_numpy(), sub[metric].to_numpy()
    H1,V1 = series("MAE","overall_all"); H2,V2 = series("MAE","hotday_all")
    H3,V3 = series("RMSE","overall_all"); H4,V4 = series("RMSE","hotday_all")
    H5,V5 = series("MAPE","overall_all"); H6,V6 = series("MAPE","hotday_all")
    H7,V7 = series("Bias","overall_all"); H8,V8 = series("Bias","hotday_all")
    fig, axs = plt.subplots(2,2, figsize=(11,7)); fig.suptitle(f"{display_name} — overall vs hotday ({tag})", fontsize=15, fontweight="bold", y=0.98)
    def fmt(ax,t,xl=None,yl=None): ax.set_title(t, fontsize=13, fontweight="bold"); ax.grid(True, alpha=.3); ax.set_xlim(1,48);
    axs[0,0].plot(H1,V1,lw=2,label="overall_all"); axs[0,0].plot(H2,V2,lw=2,ls="--",label="hotday_all"); fmt(axs[0,0],"MAE by Horizon (MW)"); axs[0,0].legend(frameon=False, fontsize=9)
    axs[0,1].plot(H3,V3,lw=2); axs[0,1].plot(H4,V4,lw=2,ls="--"); fmt(axs[0,1],"RMSE by Horizon (MW)")
    axs[1,0].plot(H5,V5,lw=2); axs[1,0].plot(H6,V6,lw=2,ls="--"); fmt(axs[1,0],"MAPE by Horizon (%)"); axs[1,0].set_xlabel("Horizon (slots ahead)"); axs[1,0].set_ylabel("MAPE (%)")
    axs[1,1].plot(H7,V7,lw=2); axs[1,1].plot(H8,V8,lw=2,ls="--"); fmt(axs[1,1],"Bias by Horizon (MW)"); axs[1,1].set_xlabel("Horizon (slots ahead)"); axs[1,1].set_ylabel("Bias (MW)")
    fig.tight_layout(rect=[0,0,1,0.95]); out = OUT_DIR / f"{display_name.lower()}__overall_vs_hotday__fourpanel__24match.png"
    fig.savefig(out, dpi=200, bbox_inches="tight"); plt.close(fig)
    print("Saved:", out.resolve()); return True

# 1) Look for any metrics CSV with EVL or Ensemble
cands = list(METRICS_DIR.glob("*evl*.csv")) + list(METRICS_DIR.glob("*ensemble*.csv")) + [METRICS_DIR/"horizon_metrics__cohort3.csv"]
print("Checked files:"); pprint([p.name for p in cands if p.exists()])

# 2) Try each, plot EVL if present
done = False
for p in cands:
    if not p.exists(): continue
    try:
        df_e = pd.read_csv(p)
        required = {"MAE","RMSE","MAPE","Bias","cohort","model","horizon"}
        if required.issubset(set(df_e.columns)):
            if plot_model_from_df(df_e, "evl", "EVL"):
                done = True; break
    except Exception as e:
        print(f"[skip] {p.name}: {e}")

if not done:
    print("No EVL metrics found in metrics folder — we’ll need ground truth to compute EVL curves.")


Checked files:
['horizon_metrics__cohort3.csv']
No EVL metrics found in metrics folder — we’ll need ground truth to compute EVL curves.


In [87]:
import pandas as pd
import numpy as np
from pathlib import Path

BASE = Path("/content/drive/MyDrive/JEFF UNI/week 6/Collab")
overall_src = BASE / "3cohort_overall_all.csv"
hotday_src  = BASE / "3cohort_hotday_all.csv"

# Load
df_overall = pd.read_csv(overall_src)
df_hot     = pd.read_csv(hotday_src)

# Extract the "true" column
y_overall_series = df_overall["true"].to_numpy()
y_hot_series     = df_hot["true"].to_numpy()

# Function: turn 1D truth series into sliding windows of length 48
def build_truth_matrix(y, horizon=48):
    N = len(y) - horizon + 1
    out = np.zeros((N, horizon))
    for i in range(N):
        out[i,:] = y[i:i+horizon]
    return out

overall_truth = build_truth_matrix(y_overall_series, 48)
hotday_truth  = build_truth_matrix(y_hot_series, 48)

# Save to Cohort3-Predictions folder
outdir = BASE / "Cohort3-Predictions"
outdir.mkdir(parents=True, exist_ok=True)
cols = [f"t+{i}" for i in range(1,49)]

pd.DataFrame(overall_truth, columns=cols).to_csv(outdir/"y_true__overall_24match.csv", index=False)
pd.DataFrame(hotday_truth,  columns=cols).to_csv(outdir/"y_true__hotday_24match.csv", index=False)

print("Saved truth CSVs to:", outdir)
print("overall_truth shape:", overall_truth.shape)
print("hotday_truth shape:", hotday_truth.shape)


Saved truth CSVs to: /content/drive/MyDrive/JEFF UNI/week 6/Collab/Cohort3-Predictions
overall_truth shape: (38689, 48)
hotday_truth shape: (3841, 48)


In [88]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from pprint import pprint

METRICS_DIR = Path("/content/drive/MyDrive/JEFF UNI/week 6/Collab/Cohort3-FINAL/Cohort3-Metrics (CSV)")
OUT_DIR = Path("figures_for_report"); OUT_DIR.mkdir(exist_ok=True, parents=True)

def plot_model_from_df(df, model_key, display_name, tag="24match"):
    df = df.copy()
    df["cohort_norm"] = df["cohort"].astype(str).str.lower().map(lambda s: "hotday_all" if "hot" in s else "overall_all")
    df["model_norm"]  = df["model"].astype(str).str.lower().map(lambda s:
        "operator" if "oper" in s else ("lightgbm" if "lightgbm" in s or "lgbm" in s else ("evl" if "evl" in s or "ensemble" in s else s))
    )
    mdf = df[df["model_norm"]==model_key]
    if mdf.empty:
        return False
    def series(metric, cohort):
        sub = mdf[(mdf["cohort_norm"]==cohort)].sort_values("horizon")
        return sub["horizon"].to_numpy(), sub[metric].to_numpy()
    H1,V1 = series("MAE","overall_all"); H2,V2 = series("MAE","hotday_all")
    H3,V3 = series("RMSE","overall_all"); H4,V4 = series("RMSE","hotday_all")
    H5,V5 = series("MAPE","overall_all"); H6,V6 = series("MAPE","hotday_all")
    H7,V7 = series("Bias","overall_all"); H8,V8 = series("Bias","hotday_all")
    fig, axs = plt.subplots(2,2, figsize=(11,7)); fig.suptitle(f"{display_name} — overall vs hotday ({tag})", fontsize=15, fontweight="bold", y=0.98)
    def fmt(ax,t,xl=None,yl=None): ax.set_title(t, fontsize=13, fontweight="bold"); ax.grid(True, alpha=.3); ax.set_xlim(1,48);
    axs[0,0].plot(H1,V1,lw=2,label="overall_all"); axs[0,0].plot(H2,V2,lw=2,ls="--",label="hotday_all"); fmt(axs[0,0],"MAE by Horizon (MW)"); axs[0,0].legend(frameon=False, fontsize=9)
    axs[0,1].plot(H3,V3,lw=2); axs[0,1].plot(H4,V4,lw=2,ls="--"); fmt(axs[0,1],"RMSE by Horizon (MW)")
    axs[1,0].plot(H5,V5,lw=2); axs[1,0].plot(H6,V6,lw=2,ls="--"); fmt(axs[1,0],"MAPE by Horizon (%)"); axs[1,0].set_xlabel("Horizon (slots ahead)"); axs[1,0].set_ylabel("MAPE (%)")
    axs[1,1].plot(H7,V7,lw=2); axs[1,1].plot(H8,V8,lw=2,ls="--"); fmt(axs[1,1],"Bias by Horizon (MW)"); axs[1,1].set_xlabel("Horizon (slots ahead)"); axs[1,1].set_ylabel("Bias (MW)")
    fig.tight_layout(rect=[0,0,1,0.95]); out = OUT_DIR / f"{display_name.lower()}__overall_vs_hotday__fourpanel__24match.png"
    fig.savefig(out, dpi=200, bbox_inches="tight"); plt.close(fig)
    print("Saved:", out.resolve()); return True

# 1) Look for any metrics CSV with EVL or Ensemble
cands = list(METRICS_DIR.glob("*evl*.csv")) + list(METRICS_DIR.glob("*ensemble*.csv")) + [METRICS_DIR/"horizon_metrics__cohort3.csv"]
print("Checked files:"); pprint([p.name for p in cands if p.exists()])

# 2) Try each, plot EVL if present
done = False
for p in cands:
    if not p.exists(): continue
    try:
        df_e = pd.read_csv(p)
        required = {"MAE","RMSE","MAPE","Bias","cohort","model","horizon"}
        if required.issubset(set(df_e.columns)):
            if plot_model_from_df(df_e, "evl", "EVL"):
                done = True; break
    except Exception as e:
        print(f"[skip] {p.name}: {e}")

if not done:
    print("No EVL metrics found in metrics folder — we’ll need ground truth to compute EVL curves.")


Checked files:
['horizon_metrics__cohort3.csv']
No EVL metrics found in metrics folder — we’ll need ground truth to compute EVL curves.


In [89]:
# === ONE CELL: make truth (from 'true' series), compute EVL metrics, plot 4-panel ===
import pandas as pd, numpy as np, matplotlib.pyplot as plt
from pathlib import Path

# ---- Paths (your Drive layout) ----
BASE = Path("/content/drive/MyDrive/JEFF UNI/week 6/Collab")
PRED_DIR = BASE / "Cohort3-Predictions"
PRED_DIR.mkdir(parents=True, exist_ok=True)
OUT_DIR = Path("figures_for_report"); OUT_DIR.mkdir(parents=True, exist_ok=True)

# ---- Load cohorts and build 48-step truth matrices from the 'true' column ----
overall_src = BASE / "3cohort_overall_all.csv"
hotday_src  = BASE / "3cohort_hotday_all.csv"

df_overall = pd.read_csv(overall_src)
df_hot     = pd.read_csv(hotday_src)

y_overall_series = df_overall["true"].to_numpy()
y_hot_series     = df_hot["true"].to_numpy()

def build_truth_matrix(y, H=48):
    N = len(y) - H + 1
    if N <= 0:
        raise ValueError(f"Series too short for {H}-step windows (len={len(y)}).")
    # Sliding window using stride trick for speed & memory
    import numpy.lib.stride_tricks as st
    stride = y.strides[0]
    mat = st.as_strided(y, shape=(N, H), strides=(stride, stride)).copy()
    return mat

overall_truth = build_truth_matrix(y_overall_series, 48)
hotday_truth  = build_truth_matrix(y_hot_series,   48)

# Save truth CSVs (also handy later)
cols48 = [f"t+{i}" for i in range(1,49)]
(pd.DataFrame(overall_truth, columns=cols48)
   .to_csv(PRED_DIR/"y_true__overall_24match.csv", index=False))
(pd.DataFrame(hotday_truth,  columns=cols48)
   .to_csv(PRED_DIR/"y_true__hotday_24match.csv",  index=False))

print("✅ Saved truth CSVs:")
print(" -", (PRED_DIR/"y_true__overall_24match.csv").name, overall_truth.shape)
print(" -", (PRED_DIR/"y_true__hotday_24match.csv").name,  hotday_truth.shape)

# ---- Load EVL predictions (your files are headerless 48-col) ----
evl_overall_fp = PRED_DIR / "predictions__JEFF__EVL__overall_all.csv"
evl_hot_fp     = PRED_DIR / "predictions__JEFF__EVL__hotday_all.csv"

y_evl_overall = pd.read_csv(evl_overall_fp, header=None).to_numpy()
y_evl_hot     = pd.read_csv(evl_hot_fp,     header=None).to_numpy()

assert y_evl_overall.shape[1] == 48 and y_evl_hot.shape[1] == 48, "EVL CSVs must have 48 columns."

# ---- Align lengths (trim to min rows so shapes match) ----
n_overall = min(overall_truth.shape[0], y_evl_overall.shape[0])
n_hot     = min(hotday_truth.shape[0],  y_evl_hot.shape[0])
y_true_overall, y_evl_overall = overall_truth[:n_overall], y_evl_overall[:n_overall]
y_true_hot,     y_evl_hot     = hotday_truth[:n_hot],      y_evl_hot[:n_hot]

# ---- Metrics by horizon ----
def horizon_metrics(y_true, y_pred):
    y_true = np.asarray(y_true); y_pred = np.asarray(y_pred)
    mae  = np.mean(np.abs(y_pred - y_true), axis=0)
    rmse = np.sqrt(np.mean((y_pred - y_true)**2, axis=0))
    mape = np.mean(np.abs((y_pred - y_true) / (np.abs(y_true) + 1e-9))*100.0, axis=0)
    bias = np.mean((y_pred - y_true), axis=0)
    return {"MAE": mae, "RMSE": rmse, "MAPE": mape, "Bias": bias}

evl_overall = horizon_metrics(y_true_overall, y_evl_overall)
evl_hot     = horizon_metrics(y_true_hot,     y_evl_hot)

# ---- Plot EVL 4-panel (same style you used for Operator/LGBM) ----
def _fmt(ax, title=None, xlabel=None, ylabel=None):
    if title:  ax.set_title(title, fontsize=13, fontweight="bold", pad=6)
    if xlabel: ax.set_xlabel(xlabel, fontsize=10)
    if ylabel: ax.set_ylabel(ylabel, fontsize=10)
    ax.grid(True, alpha=0.3); ax.set_xlim(1, 48)

H = np.arange(1,49)
fig, axs = plt.subplots(2, 2, figsize=(11, 7))
fig.suptitle("EVL — overall vs hotday (24match)", fontsize=15, fontweight="bold", y=0.98)

axs[0,0].plot(H, evl_overall["MAE"],  lw=2, label="overall_all")
axs[0,0].plot(H, evl_hot["MAE"],      lw=2, ls="--", label="hotday_all")
_fmt(axs[0,0], "MAE by Horizon (MW)", None, "MAE (MW)"); axs[0,0].legend(frameon=False, fontsize=9)

axs[0,1].plot(H, evl_overall["RMSE"], lw=2)
axs[0,1].plot(H, evl_hot["RMSE"],     lw=2, ls="--")
_fmt(axs[0,1], "RMSE by Horizon (MW)", None, "RMSE (MW)")

axs[1,0].plot(H, evl_overall["MAPE"], lw=2)
axs[1,0].plot(H, evl_hot["MAPE"],     lw=2, ls="--")
_fmt(axs[1,0], "MAPE by Horizon (%)", "Horizon (slots ahead)", "MAPE (%)")

axs[1,1].plot(H, evl_overall["Bias"], lw=2)
axs[1,1].plot(H, evl_hot["Bias"],     lw=2, ls="--")
_fmt(axs[1,1], "Bias by Horizon (pred − true, MW)", "Horizon (slots ahead)", "Bias (MW)")

fig.tight_layout(rect=[0,0,1,0.95])
out = OUT_DIR / "evl__overall_vs_hotday__fourpanel__24match.png"
fig.savefig(out, dpi=200, bbox_inches="tight"); plt.close(fig)
print("✅ Saved:", out.resolve())


✅ Saved truth CSVs:
 - y_true__overall_24match.csv (38689, 48)
 - y_true__hotday_24match.csv (3841, 48)
✅ Saved: /content/figures_for_report/evl__overall_vs_hotday__fourpanel__24match.png


In [91]:
# === Stitch Operator, LGBM, EVL 4-panels into one figure (PNG + PDF) ===
from pathlib import Path
from PIL import Image, ImageDraw, ImageFont

FIGDIR = Path("figures_for_report")
paths = [
    FIGDIR / "operator__overall_vs_hotday__fourpanel__24match.png",
    FIGDIR / "lgbm__overall_vs_hotday__fourpanel__24match.png",
    FIGDIR / "evl__overall_vs_hotday__fourpanel__24match.png",
]

# Load images and standardize width
imgs = []
for p in paths:
    if not p.exists():
        raise FileNotFoundError(f"Missing figure: {p}. Generate it first.")
    imgs.append(Image.open(p).convert("RGB"))

# Target width = min width among images (to avoid upscaling)
target_w = min(im.size[0] for im in imgs)
scaled = []
for im in imgs:
    w, h = im.size
    if w != target_w:
        new_h = int(h * (target_w / w))
        im = im.resize((target_w, new_h), Image.LANCZOS)
    scaled.append(im)

labels = ["Operator", "LGBM", "EVL"]

spacer = 40
label_h = 50
sep_color = (240, 240, 240)
label_bg = (250, 250, 250)
text_color = (30, 30, 30)

try:
    font = ImageFont.truetype("DejaVuSans-Bold.ttf", 28)
except:
    font = ImageFont.load_default()

blocks = []
for label, im in zip(labels, scaled):
    block = Image.new("RGB", (target_w, label_h + im.size[1]), label_bg)
    draw = ImageDraw.Draw(block)
    # Center label text (use textbbox instead of textsize)
    bbox = draw.textbbox((0, 0), label, font=font)
    tw, th = bbox[2] - bbox[0], bbox[3] - bbox[1]
    draw.text(((target_w - tw)//2, (label_h - th)//2), label, fill=text_color, font=font)
    block.paste(im, (0, label_h))
    blocks.append(block)

total_h = sum(b.size[1] for b in blocks) + spacer * (len(blocks) - 1)
canvas = Image.new("RGB", (target_w, total_h), "white")

y = 0
for i, b in enumerate(blocks):
    canvas.paste(b, (0, y))
    y += b.size[1]
    if i < len(blocks) - 1:
        ImageDraw.Draw(canvas).rectangle([0, y, target_w, y + spacer], fill=sep_color)
        y += spacer

combined_png = FIGDIR / "combined__operator_lgbm_evl__fourpanel__24match.png"
combined_pdf = FIGDIR / "combined__operator_lgbm_evl__fourpanel__24match.pdf"
canvas.save(combined_png, "PNG")
canvas.save(combined_pdf, "PDF")
print("Saved:")
print(" -", combined_png.resolve())
print(" -", combined_pdf.resolve())


Saved:
 - /content/figures_for_report/combined__operator_lgbm_evl__fourpanel__24match.png
 - /content/figures_for_report/combined__operator_lgbm_evl__fourpanel__24match.pdf


In [1]:
# === EVL @ t+48: compute MAE/RMSE/MAPE/Bias from week 6 folder (Cohort-3 only) ===
# - Finds EVL preds in:   JEFF UNI/week 6/Collab/Cohort3-Predictions/
# - Finds truths in:      same folder (y_true__*.csv) OR rebuilds from 3cohort_*_all.csv 'true' series
# - Prints t+48 metrics for overall_all and hotday_all, and saves a CSV summary.

import os, numpy as np, pandas as pd
from pathlib import Path

# 0) Mount Drive if needed
try:
    from google.colab import drive  # will error locally; fine in Colab
    if not Path("/content/drive").exists():
        pass
    elif not Path("/content/drive/MyDrive").exists():
        drive.mount('/content/drive')
except Exception:
    # not in Colab, ignore
    pass

# 1) Paths
BASE = Path("/content/drive/MyDrive/JEFF UNI/week 6/Collab")
PRED_DIR = BASE / "Cohort3-Predictions"
FIG_DIR  = Path("figures_for_report")
FIG_DIR.mkdir(parents=True, exist_ok=True)

evl_overall_fp = PRED_DIR / "predictions__JEFF__EVL__overall_all.csv"
evl_hot_fp     = PRED_DIR / "predictions__JEFF__EVL__hotday_all.csv"

truth_overall_cands = [
    PRED_DIR / "y_true__overall_24match.csv",
    PRED_DIR / "truth__overall_24match.csv",
    PRED_DIR / "y_true__overall_all.csv",
]
truth_hot_cands = [
    PRED_DIR / "y_true__hotday_24match.csv",
    PRED_DIR / "truth__hotday_24match.csv",
    PRED_DIR / "y_true__hotday_all.csv",
]

# 2) Helpers
def load_evl_preds(path):
    """
    Your EVL CSVs are headerless 48-col matrices. This loader enforces that.
    """
    if not path.exists():
        raise FileNotFoundError(f"Missing EVL file: {path}")
    df = pd.read_csv(path, header=None)
    if df.shape[1] != 48:
        raise ValueError(f"{path.name}: expected 48 columns (horizons), got {df.shape[1]}")
    return df.to_numpy()

def load_truth_from_candidates(cands):
    for p in cands:
        if p.exists():
            df = pd.read_csv(p)
            cols = [f"t+{i}" for i in range(1,49)]
            if all(c in df.columns for c in cols):
                return df[cols].to_numpy(), p
            if df.shape[1] == 48:
                return df.iloc[:, :48].to_numpy(), p
    return None, None

def rebuild_truth_from_3cohort(csv_path):
    """
    Build (N,48) truth matrix using sliding 48-slot windows from the 'true' column
    of 3cohort_*_all.csv (Cohort-3 only).
    """
    df = pd.read_csv(csv_path)
    if "true" not in df.columns:
        raise ValueError(f"'true' column not found in {csv_path.name}")
    y = df["true"].to_numpy()
    H = 48
    N = len(y) - H + 1
    if N <= 0:
        raise ValueError(f"{csv_path.name}: series too short to form 48-slot windows (len={len(y)})")
    # fast sliding windows
    import numpy.lib.stride_tricks as st
    stride = y.strides[0]
    mat = st.as_strided(y, shape=(N, H), strides=(stride, stride)).copy()
    return mat, csv_path

def align_for_eval(y_true, y_pred):
    n = min(y_true.shape[0], y_pred.shape[0])
    return y_true[:n], y_pred[:n]

def metrics_by_horizon(y_true, y_pred):
    err = y_pred - y_true
    mae  = np.mean(np.abs(err), axis=0)
    rmse = np.sqrt(np.mean(err**2, axis=0))
    mape = np.mean(np.abs(err) / (np.abs(y_true) + 1e-9), axis=0) * 100.0
    bias = np.mean(err, axis=0)
    return dict(MAE=mae, RMSE=rmse, MAPE=mape, Bias=bias)

def print_block(title, d):
    print("\n" + "="*len(title))
    print(title)
    print("="*len(title))
    for k, v in d.items():
        print(f"{k}: {v}")

# 3) Load EVL forecasts
y_evl_overall = load_evl_preds(evl_overall_fp)
y_evl_hot     = load_evl_preds(evl_hot_fp)

# 4) Load truths (prefer ready-made CSVs; else rebuild from Cohort-3 3cohort_*_all.csv)
y_true_overall, src_overall = load_truth_from_candidates(truth_overall_cands)
if y_true_overall is None:
    y_true_overall, src_overall = rebuild_truth_from_3cohort(BASE / "3cohort_overall_all.csv")

y_true_hot, src_hot = load_truth_from_candidates(truth_hot_cands)
if y_true_hot is None:
    y_true_hot, src_hot = rebuild_truth_from_3cohort(BASE / "3cohort_hotday_all.csv")

# 5) Align shapes (trim to common N if needed)
y_true_overall, y_evl_overall = align_for_eval(y_true_overall, y_evl_overall)
y_true_hot,     y_evl_hot     = align_for_eval(y_true_hot,     y_evl_hot)

# 6) Compute metrics
evl_overall = metrics_by_horizon(y_true_overall, y_evl_overall)
evl_hot     = metrics_by_horizon(y_true_hot,     y_evl_hot)

# 7) Extract t+48 (column index 47)
def t48_row(metrics):
    return { m: float(np.round(metrics[m][47], 6)) for m in ["MAE","RMSE","MAPE","Bias"] }

t48_overall = t48_row(evl_overall)
t48_hot     = t48_row(evl_hot)

# 8) Print nicely (chunked)
print_block("EVL @ t+48 — overall_all (Cohort-3)", t48_overall)
print(f"Truth source: {src_overall}")
print_block("EVL @ t+48 — hotday_all (Cohort-3)", t48_hot)
print(f"Truth source: {src_hot}")

# 9) Save a tiny CSV near your figures (so Maria can file it)
out = pd.DataFrame([
    dict(Model="EVL", Cohort="overall_all", **t48_overall),
    dict(Model="EVL", Cohort="hotday_all", **t48_hot),
])
out_path = FIG_DIR / "t48_evl_metrics__from_week6.csv"
out.to_csv(out_path, index=False)
print("\nSaved:", out_path.resolve())



EVL @ t+48 — overall_all (Cohort-3)
MAE: 1543.081086
RMSE: 1935.799816
MAPE: 17.567778
Bias: -179.995765
Truth source: /content/drive/MyDrive/JEFF UNI/week 6/Collab/Cohort3-Predictions/y_true__overall_24match.csv

EVL @ t+48 — hotday_all (Cohort-3)
MAE: 1797.173648
RMSE: 2174.647752
MAPE: 23.267425
Bias: 937.238833
Truth source: /content/drive/MyDrive/JEFF UNI/week 6/Collab/Cohort3-Predictions/y_true__hotday_24match.csv

Saved: /content/figures_for_report/t48_evl_metrics__from_week6.csv


In [2]:
# === Cohort-3 strict alignment & EVL metrics @ t+48 (no ±5 min, no guessing) ===
import pandas as pd, numpy as np
from pathlib import Path

BASE = Path("/content/drive/MyDrive/JEFF UNI/week 6/Collab")
PRED_DIR = BASE / "Cohort3-Predictions"

# 1) Load Cohort-3 CSVs and enforce a strict 30-minute timeline
def load_cohort_truth_matrix(csv_path):
    df = pd.read_csv(csv_path)
    # choose exact column names—do NOT auto-detect
    time_col = "timestamp" if "timestamp" in df.columns else (
        "trading_interval" if "trading_interval" in df.columns else None
    )
    if time_col is None:
        raise ValueError(f"{csv_path.name}: expected 'timestamp' or 'trading_interval' column.")
    if "true" not in df.columns:
        raise ValueError(f"{csv_path.name}: missing 'true' target column.")

    # standardize to UTC, round to 30m, drop dupes, sort
    t = pd.to_datetime(df[time_col], errors="raise", utc=True).round("30min")
    s = pd.Series(df["true"].to_numpy(), index=t).sort_index()
    s = s[~s.index.duplicated(keep="first")]

    # build complete 30-min time index across span, then reindex (strict)
    full_idx = pd.date_range(s.index.min(), s.index.max(), freq="30min", tz="UTC")
    s = s.reindex(full_idx)

    # build 48-slot daily windows starting at midnight UTC boundaries in this index
    # window_id = timestamp of the first slot
    starts = s.index[(s.index.minute==0) & (s.index.hour==0)]  # daily midnights UTC
    rows, ids = [], []
    for start in starts:
        end = start + pd.Timedelta(hours=24)
        slice_ = s.loc[start:end - pd.Timedelta(minutes=30)]
        if len(slice_) == 48 and slice_.isna().sum() == 0:
            rows.append(slice_.to_numpy())
            ids.append(start)  # window_id

    if not rows:
        raise ValueError(f"{csv_path.name}: no complete 48-slot days found on strict grid.")
    Y = np.vstack(rows)  # (N,48)
    return Y, pd.to_datetime(ids)

overall_truth, overall_ids = load_cohort_truth_matrix(BASE / "3cohort_overall_all.csv")
hot_truth, hot_ids         = load_cohort_truth_matrix(BASE / "3cohort_hotday_all.csv")

print("Cohort-3 windows (strict): overall N=", overall_truth.shape[0], "hot N=", hot_truth.shape[0])

# 2) Load EVL predictions (headerless 48-cols) and assert same N
def load_evl_preds(path):
    df = pd.read_csv(path, header=None)
    if df.shape[1] != 48:
        raise ValueError(f"{path.name}: expected 48 columns, got {df.shape[1]}")
    return df.to_numpy()

evl_overall = load_evl_preds(PRED_DIR / "predictions__JEFF__EVL__overall_all.csv")
evl_hot     = load_evl_preds(PRED_DIR / "predictions__JEFF__EVL__hotday_all.csv")

print("EVL preds: overall N=", evl_overall.shape[0], "hot N=", evl_hot.shape[0])

# Strict count check — refuse to compute if N differs (prevents silent drift)
if evl_overall.shape[0] != overall_truth.shape[0]:
    raise RuntimeError(f"overall N mismatch: EVL {evl_overall.shape[0]} vs truth {overall_truth.shape[0]}")
if evl_hot.shape[0] != hot_truth.shape[0]:
    raise RuntimeError(f"hotday N mismatch: EVL {evl_hot.shape[0]} vs truth {hot_truth.shape[0]}")

# 3) Metrics by horizon
def metrics_by_horizon(y_true, y_pred):
    err = y_pred - y_true
    mae  = np.mean(np.abs(err), axis=0)
    rmse = np.sqrt(np.mean(err**2, axis=0))
    mape = np.mean(np.abs(err) / (np.abs(y_true) + 1e-9), axis=0) * 100.0
    bias = np.mean(err, axis=0)
    return dict(MAE=mae, RMSE=rmse, MAPE=mape, Bias=bias)

m_overall = metrics_by_horizon(overall_truth, evl_overall)
m_hot     = metrics_by_horizon(hot_truth,     evl_hot)

# 4) Report t+48 only (index 47)
def t48(d):
    return {k: float(np.round(d[k][47], 6)) for k in ["MAE","RMSE","MAPE","Bias"]}

print("\nEVL @ t+48 (STRICT, Cohort-3 exact windows)")
print("overall_all:", t48(m_overall))
print("hotday_all :", t48(m_hot))


Cohort-3 windows (strict): overall N= 807 hot N= 81
EVL preds: overall N= 807 hot N= 81

EVL @ t+48 (STRICT, Cohort-3 exact windows)
overall_all: {'MAE': 995.308298, 'RMSE': 1215.797162, 'MAPE': 13.335763, 'Bias': 940.992438}
hotday_all : {'MAE': 1424.989878, 'RMSE': 1711.171381, 'MAPE': 18.528216, 'Bias': 1397.568463}


In [3]:
# Zip the figures folder and download locally
import shutil, os
from google.colab import files

FOLDER = "figures_for_report"  # change if needed
assert os.path.isdir(FOLDER), f"Folder not found: {FOLDER}"

zip_name = f"{FOLDER}.zip"
shutil.make_archive(FOLDER, "zip", FOLDER)
files.download(zip_name)  # prompts a download to your computer


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [4]:
from google.colab import files
files.download("figures_for_report/t48_evl_metrics__from_week6.csv")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [5]:
# If a file lives on Drive, copy it to /content first, then download
import shutil, os
from google.colab import files
from pathlib import Path

drive_path = Path("/content/drive/MyDrive/JEFF UNI/week 6/Collab/Cohort3-Predictions/y_true__overall_24match.csv")
local_copy = Path("/content") / drive_path.name

shutil.copy(drive_path, local_copy)
files.download(str(local_copy))


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [6]:
# Bundle a list of specific files (from content or Drive) into a single ZIP
import shutil, os
from google.colab import files
from pathlib import Path

want = [
    "figures_for_report/operator__overall_vs_hotday__fourpanel__24match.png",
    "figures_for_report/lgbm__overall_vs_hotday__fourpanel__24match.png",
    "figures_for_report/evl__overall_vs_hotday__fourpanel__24match.png",
    "figures_for_report/combined__operator_lgbm_evl__fourpanel__24match.png",
    "figures_for_report/t48_evl_metrics__from_week6.csv",
]

# Verify all exist; copy any Drive files to /content first if needed
for p in want:
    if not os.path.exists(p):
        print("Missing:", p)

zip_name = "submission_bundle.zip"
shutil.make_archive(zip_name.replace(".zip",""), "zip", root_dir="/content", base_dir="figures_for_report")
from google.colab import files
files.download(zip_name)


Missing: figures_for_report/operator__overall_vs_hotday__fourpanel__24match.png
Missing: figures_for_report/lgbm__overall_vs_hotday__fourpanel__24match.png
Missing: figures_for_report/evl__overall_vs_hotday__fourpanel__24match.png
Missing: figures_for_report/combined__operator_lgbm_evl__fourpanel__24match.png


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
[notebooks] found=0 copied=0 -> /content/drive/MyDrive/LGBM_EVL_C3_Jeff/1_notebooks
[training_data] found=6 copied=6 -> /content/drive/MyDrive/LGBM_EVL_C3_Jeff/2_training_data
[models] found=20 copied=20 -> /content/drive/MyDrive/LGBM_EVL_C3_Jeff/3_models
[tables] found=10 copied=10 -> /content/drive/MyDrive/LGBM_EVL_C3_Jeff/4_tables
[figures] found=87 copied=87 -> /content/drive/MyDrive/LGBM_EVL_C3_Jeff/5_figures
[outputs] found=0 copied=0 -> /content/drive/MyDrive/LGBM_EVL_C3_Jeff/6_outputs

⚠️ Notebook not found. Please ensure your notebook is named 'LGBM_EVL_C3_Jeff.ipynb' or one of:
   - Collab3_Jeff.ipynb
   - Collab3_Jeff (1).ipynb
   - Collab3_Jeff (2).ipynb
Then re-run this cell.
Included local figures folder: /content/drive/MyDrive/LGBM_EVL_C3_Jeff/5_figures/figures_for_report

✅ Done. Master folder: /content/drive/MyDrive/LGBM_EVL_C3_Jeff
 - notebo